# Building age — ground-up rebuild

Input: the raw Madrid and Amsterdam parquets only.
1. Madrid preprocessing (v3 rules, 10 spatial folds, neighbourhood channels), checksums vs previous build
2. Backbone: 10-seed TempCNN ensemble, Madrid 10-fold CV + models on all Madrid
3. Pre-registered rules on this CV: best checkpoint vs SWA, number of members
4. Transfer evaluation (modelCreate) on Amsterdam: k = 5–100 labels per class, 10 draws each

In [ ]:
import glob, os, sys, json, time, subprocess, shutil
import torch
print('CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available()
WD = '/kaggle/working'
for d in ('model_selection_v3', 'model_selection_final', 'hackathon/data', 'logs'):
    os.makedirs(f'{WD}/{d}', exist_ok=True)
for n in ('madrid_train.parquet', 'amsterdam_data.parquet'):
    f = glob.glob(f'/kaggle/input/**/{n}', recursive=True); assert f, n
    os.path.exists(f'{WD}/hackathon/data/{n}') or os.symlink(f[0], f'{WD}/hackathon/data/{n}')
ENV = dict(os.environ, MS_CACHE_DIR=f'{WD}/cache', MS_EXTRA_DIR=f'{WD}/extra', MS_RESULTS_DIR=f'{WD}/results',
           N_THREADS='2', OMP_NUM_THREADS='2')
PY = [sys.executable, '-W', 'ignore']
def run_parallel(jobs, tag):
    procs = []
    for i, (args, env) in enumerate(jobs, 1):
        log = open(f'{WD}/logs/{tag}_w{i}.log', 'w')
        procs.append(subprocess.Popen(PY + args, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=WD)); time.sleep(15)
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(120); print(f'[{tag} {(time.time() - t0) / 60:5.1f} min] running', flush=True)
    for i, p in enumerate(procs, 1):
        print(f'{tag} worker {i} exit {p.returncode}'); print(''.join(open(f'{WD}/logs/{tag}_w{i}.log').readlines()[-4:]))
        assert p.returncode == 0, f'{tag} worker {i} failed'

In [ ]:
%%writefile /kaggle/working/model_selection_v3/common_v3.py
"""
v3 inputs -- the rules agreed after the checks (see model_selection_v2/checks_v3.py).

Observation choice (per pixel, year, band), using ONLY each observation's own properties,
never values from other years (no temporal smoothing, nothing deleted):
  usable  : QA-valid (pipeline `qa_valid`: clear, no fill / cloud / shadow / dilated cloud)
            AND the band is not saturated: QA_RADSAT bit of that band (sensor-specific)
            OR the value sits at that sensor+band's encoding ceiling (data-driven: the top
            value where >= 10 observations pile up -- catches the corrupt 2002 Amsterdam scene)
  ranking : whole acquisitions are ranked, so the 6 bands share a date unless a band is
            unusable in the best one:
              1. not an anomalous acquisition (city-wide median reflectance is a robust
                 outlier, z > 3, among the city's own acquisitions -- same for every pixel)
              2. lower QA cloud confidence   3. Collection-2 Tier 1 before Tier 2
              4. day of year closest to mid-summer (tie-break only)
  no usable observation in a year -> NB3 interpolation, flagged by the observed mask.
Removed vs v2: Blue > 15,000 rule, temporal outlier test, HOT haze score, pixel_x/pixel_y,
the 12 per-pixel metadata averages, raw QA channels. Added: distance to centre.
Label weights (training only): heaped construction years (detected from each city's own
histogram) and closeness to a class boundary lower a pixel's weight (floor 0.2).
weighted_mean_year is never an input.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

ROOT        = Path(__file__).resolve().parent
DATA_DIR    = ROOT.parent / "hackathon" / "data"            # never the held-out folder
CACHE_DIR   = Path(os.environ.get("MS_CACHE_DIR", ROOT / "cache"))
RESULTS_DIR = Path(os.environ.get("MS_RESULTS_DIR", ROOT / "results"))
CITY_FILES  = {"Madrid": "madrid_train.parquet", "Amsterdam": "amsterdam_data.parquet"}

BASE_BANDS   = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]
INDEX_COLS   = ["NDVI", "NDBI", "UI", "MNDWI", "BSI"]
SENSORS      = ["LT04", "LT05", "LE07", "LC08", "LC09"]
CLASS_EVENTS = {"Amsterdam": 1945, "Madrid": 1960}
EARLY_CUTOFF, LATE_START = 2003, 2004
RADSAT_BIT   = {"TM": [0, 1, 2, 3, 4, 6], "OLI": [1, 2, 3, 4, 5, 6]}   # USGS C2 L2 QA_RADSAT
CENTRES      = {"Madrid": (40.4169, -3.7035, 30), "Amsterdam": (52.3731, 4.8932, 31)}  # Sol / Dam

MID_SUMMER_DOY, Z_ANOM, CEIL_MIN_COUNT, MIN_ACQ_OBS = 196, 3.0, 10, 100
W_MIN, TAU, TAU_HEAP, Z_HEAP = 0.2, 1.0, 5.0, 3.0

META_CHANNELS = ([f"obs_{b}" for b in BASE_BANDS] + ["doy"] + [f"sensor_{s}" for s in SENSORS]
                 + ["n_valid_slots", "anomalous_acq", "tier2"])
SEQ_CHANNELS = BASE_BANDS + INDEX_COLS + META_CHANNELS                      # 11 + 15 = 26
STATIC_COLS  = ["coverage", "dist_centre_rel"]
NB3_FEATURES = ([f"{c}_mean" for c in BASE_BANDS] + [f"{c}_std" for c in BASE_BANDS]
                + [f"{c}_mean" for c in INDEX_COLS] + [f"{c}_std" for c in INDEX_COLS]
                + [f"{b}_early_mean" for b in BASE_BANDS] + [f"{b}_early_std" for b in BASE_BANDS]
                + [f"{b}_late_mean" for b in BASE_BANDS] + [f"{b}_late_std" for b in BASE_BANDS]
                + [f"d_{b}_mean" for b in BASE_BANDS] + [f"d_{b}_std" for b in BASE_BANDS]
                + ["has_early_data", "has_late_data"])
TAB_FEATURES = NB3_FEATURES + STATIC_COLS                                   # 60 + 2 = 62
assert len(SEQ_CHANNELS) == 26 and len(TAB_FEATURES) == 62
assert not {"weighted_mean_year", "pixel_x", "pixel_y"} & set(TAB_FEATURES + SEQ_CHANNELS)


def compute_indices(B, G, R, N, S1, S2):
    eps = 1e-6
    return [(N - R) / (N + R + eps), (S1 - N) / (S1 + N + eps), (S2 - N) / (S2 + N + eps),
            (G - S1) / (G + S1 + eps), ((S1 + R) - (N + B)) / ((S1 + R) + (N + B) + eps)]


def utm(lat, lon, zone=None):
    """WGS84 -> UTM, standard transverse-Mercator series. Zone from the longitude if not given;
    southern hemisphere uses the 10,000 km false northing."""
    if zone is None:
        zone = int((lon + 180) // 6) + 1
    south = lat < 0
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996
    e2 = f * (2 - f); ep2 = e2 / (1 - e2)
    lat, lon0 = np.radians(lat), np.radians((zone - 1) * 6 - 180 + 3)
    N = a / np.sqrt(1 - e2 * np.sin(lat) ** 2); T = np.tan(lat) ** 2; C = ep2 * np.cos(lat) ** 2
    A = np.cos(lat) * (np.radians(lon) - lon0)
    M = a * ((1 - e2 / 4 - 3 * e2 ** 2 / 64 - 5 * e2 ** 3 / 256) * lat
             - (3 * e2 / 8 + 3 * e2 ** 2 / 32 + 45 * e2 ** 3 / 1024) * np.sin(2 * lat)
             + (15 * e2 ** 2 / 256 + 45 * e2 ** 3 / 1024) * np.sin(4 * lat)
             - (35 * e2 ** 3 / 3072) * np.sin(6 * lat))
    x = k0 * N * (A + (1 - T + C) * A ** 3 / 6 + (5 - 18 * T + T ** 2 + 72 * C - 58 * ep2) * A ** 5 / 120)
    y = k0 * (M + N * np.tan(lat) * (A ** 2 / 2 + (5 - T + 9 * C + 4 * C ** 2) * A ** 4 / 24
                                     + (61 - 58 * T + T ** 2 + 600 * C - 330 * ep2) * A ** 6 / 720))
    return x + 500000, (y + 10_000_000 if south else y)


def label_weights(wmy, city=None, bounds=None):
    """Training weight = reliability of the class label. Heaped years are detected from the
    dataset's own histogram; weight shrinks near any class boundary, faster for heaped years.
    `bounds`: the dataset's class boundaries (years). `city` only for the legacy v3 callers."""
    years = np.round(wmy).astype(int)
    is_int = np.isclose(wmy, years)
    counts = pd.Series(years[is_int]).value_counts()
    yr = np.arange(int(wmy.min()), int(wmy.max()) + 1)
    c = counts.reindex(yr, fill_value=0).to_numpy(float)
    ratio = np.ones_like(c)
    for i in range(len(yr)):
        nb = [c[j] for j in range(max(0, i - 3), min(len(yr), i + 4)) if j != i]
        ratio[i] = (c[i] + 1) / (np.median(nb) + 1)
    lr = np.log(ratio)
    mad = 1.4826 * np.median(np.abs(lr - np.median(lr)))
    # too few labels to estimate the histogram's spread -> no year can be called heaped
    heaped_years = yr[(lr - np.median(lr)) / mad > Z_HEAP] if mad > 0 else yr[:0]
    heaped = is_int & np.isin(years, heaped_years)
    bounds = np.array([CLASS_EVENTS[city], 1984, 2004] if bounds is None else bounds, float)
    d = np.abs(wmy[:, None] - bounds[None, :]).min(axis=1)
    tau = np.where(heaped, TAU_HEAP, TAU)
    w = W_MIN + (1 - W_MIN) * (1 - np.exp(-d / tau))
    info = dict(heaped_years={int(y): int(counts.get(y, 0)) for y in heaped_years},
                pixels_in_heaped_years=int(heaped.sum()), mean_weight=float(w.mean()),
                share_below_0_5=float((w < 0.5).mean()))
    return w.astype(np.float32), info


def preprocess_city_v3(city, return_internals=False):
    """Legacy entry point (reproduces the v3/v4 caches): the two hackathon cities."""
    lat, lon, zone = CENTRES[city]
    return preprocess_dataset(DATA_DIR / CITY_FILES[city], centre_xy=utm(lat, lon, zone),
                              class_bounds=(CLASS_EVENTS[city], 1984, 2004), return_internals=return_internals)


def preprocess_dataset(path, centre_xy=None, class_bounds=None, doy_shift=0.0, strict=True,
                       keep_xy=False, return_internals=False):
    """The v3 rules for ANY dataset in the hackathon parquet format. Nothing here depends on
    which city it is; everything city-dependent is an argument:
      centre_xy    centre in the dataset's own projected coordinates (None -> dist_centre_rel NaN)
      class_bounds the dataset's class boundary years (None or no label column -> no labels)
      doy_shift    days added to every acquisition DOY before the season rules (0 in the
                   northern hemisphere; half a year in the southern one, so 'summer' means summer)
      strict       False: a pixel-band with no usable observation in any year is filled with the
                   dataset's per-year median of observed values (still flagged unobserved)"""
    import pyarrow.parquet as pq
    have = set(pq.read_schema(path).names)
    cols = ["year", "px_key", "py_key", "pixel_x", "pixel_y", "coverage"]
    has_labels = "weighted_mean_year" in have and class_bounds is not None
    if has_labels:
        cols.append("weighted_mean_year")
    for s in (1, 2, 3):
        cols += [f"{b}_{s}" for b in BASE_BANDS]
        cols += [f"QA_PIXEL_{s}", f"QA_RADSAT_{s}", f"doy_{s}", f"scene_id_{s}", f"qa_valid_{s}"]
    missing = [c for c in cols if c not in have]
    if missing:
        raise ValueError(f"dataset lacks required columns: {missing}")
    df = pd.read_parquet(path, columns=cols)
    key = df["px_key"].to_numpy(np.int64) * 1_000_000 + df["py_key"].to_numpy(np.int64)
    ukey, first, pix = np.unique(key, return_index=True, return_inverse=True)
    y0 = int(df["year"].min()); years = np.arange(y0, int(df["year"].max()) + 1)
    t = df["year"].to_numpy() - y0
    N, T = len(ukey), len(years)

    V = np.full((N, T, 3, 6), np.nan, np.float32)
    QA = np.zeros((N, T, 3), np.int64); RAD = np.zeros((N, T, 3), np.int64)
    DOY = np.zeros((N, T, 3), np.float32); SEN = np.full((N, T, 3), -1, np.int8)
    VALID = np.zeros((N, T, 3), bool); TIER2 = np.zeros((N, T, 3), bool)
    ACQ = np.full((N, T, 3), -1, np.int32)
    scene_codes, scene_names = pd.factorize(pd.concat([df[f"scene_id_{s}"] for s in (1, 2, 3)]))
    n = len(df)
    for s in (1, 2, 3):
        i = s - 1
        for j, b in enumerate(BASE_BANDS):
            V[pix, t, i, j] = df[f"{b}_{s}"].to_numpy(np.float32, na_value=np.nan)
        QA[pix, t, i] = np.nan_to_num(df[f"QA_PIXEL_{s}"].to_numpy(float)).astype(np.int64)
        RAD[pix, t, i] = np.nan_to_num(df[f"QA_RADSAT_{s}"].to_numpy(float)).astype(np.int64)
        DOY[pix, t, i] = np.nan_to_num(df[f"doy_{s}"].to_numpy(float))
        if doy_shift:
            DOY[pix, t, i] = np.where(DOY[pix, t, i] > 0, (DOY[pix, t, i] + doy_shift - 1) % 365.25 + 1, 0)
        sid = df[f"scene_id_{s}"].fillna("")
        unknown = sorted(set(sid[(sid != "") & ~sid.str[:4].isin(SENSORS)].str[:4]))
        if unknown:
            raise ValueError(f"unknown sensor(s) {unknown}: the backbone only knows {SENSORS}")
        SEN[pix, t, i] = np.select([sid.str[:4].to_numpy() == x for x in SENSORS], range(len(SENSORS)), -1)
        TIER2[pix, t, i] = sid.str.endswith("T2").to_numpy()
        VALID[pix, t, i] = df[f"qa_valid_{s}"].fillna(False).to_numpy(bool)
        ACQ[pix, t, i] = scene_codes[i * n:(i + 1) * n]
    static_cov = df["coverage"].to_numpy(np.float64)[first]
    px_m, py_m = df["pixel_x"].to_numpy(np.float64)[first], df["pixel_y"].to_numpy(np.float64)[first]
    wmy = df["weighted_mean_year"].to_numpy(np.float64)[first] if has_labels else None
    px_key, py_key = df["px_key"].to_numpy()[first], df["py_key"].to_numpy()[first]
    del df

    # ── saturation: QA_RADSAT band bit, or value at the sensor+band encoding ceiling ────
    is_oli = SEN >= SENSORS.index("LC08")
    ceil = np.full((len(SENSORS), 6), np.inf, np.float32)
    ceil_info = {}
    for k, sname in enumerate(SENSORS):
        m = VALID & (SEN == k)
        for j, b in enumerate(BASE_BANDS):
            vals = V[..., j][m]
            vals = vals[~np.isnan(vals)]
            if len(vals) == 0:
                continue
            vmax = vals.max()
            n_at = int((vals >= vmax - 1).sum())
            if n_at >= CEIL_MIN_COUNT:
                ceil[k, j] = vmax - 1
                ceil_info[f"{sname}_{b}"] = dict(ceiling=float(vmax), n_at_ceiling=n_at)
    usable = np.zeros(V.shape, bool)
    n_sat_qa = n_sat_ceil = 0
    for j in range(6):
        sat_qa = np.where(is_oli, (RAD >> RADSAT_BIT["OLI"][j]) & 1, (RAD >> RADSAT_BIT["TM"][j]) & 1) == 1
        at_ceil = (SEN >= 0) & (V[..., j] >= ceil[np.clip(SEN, 0, None), j])
        base = VALID & ~np.isnan(V[..., j])
        n_sat_qa += int((base & sat_qa).sum()); n_sat_ceil += int((base & at_ceil & ~sat_qa).sum())
        usable[..., j] = base & ~sat_qa & ~at_ceil

    # ── anomalous acquisitions: city-wide median reflectance is a robust outlier ───────
    m = VALID & ~np.isnan(V).any(axis=3)          # all QA-valid obs, ceiling values included
    acq_obs = pd.DataFrame(V[m], columns=BASE_BANDS)
    acq_obs["acq"] = ACQ[m]
    g = acq_obs.groupby("acq")
    med = g[BASE_BANDS].median()
    med = med[g.size() >= MIN_ACQ_OBS]
    zz = (med - med.median()) / (1.4826 * (med - med.median()).abs().median())
    acq_z = zz.abs().max(axis=1)
    anom_codes = acq_z.index[acq_z > Z_ANOM].to_numpy()
    ANOM = np.isin(ACQ, anom_codes)
    anomalous_list = sorted(((str(scene_names[c]), float(acq_z[c])) for c in anom_codes),
                            key=lambda x: -x[1])

    # ── rank acquisitions, choose per band ─────────────────────────────────────────────
    cloud_med = ((QA >> 8) & 3) == 2
    rank = (ANOM * 1e6 + cloud_med * 1e4 + TIER2 * 1e2
            + np.abs(DOY - MID_SUMMER_DOY) / 1e3).astype(np.float64)
    chosen = np.full((N, T, 6), -1, np.int8)
    for j in range(6):
        r = np.where(usable[..., j], rank, np.inf)
        chosen[..., j] = np.where(np.isfinite(r.min(axis=2)), r.argmin(axis=2), -1)
    obs = chosen >= 0
    ci = np.clip(chosen, 0, 2).astype(np.int64)
    bands = np.stack([np.take_along_axis(V[..., j], ci[..., j:j + 1], axis=2)[..., 0]
                      for j in range(6)], axis=2)
    bands = np.where(obs, bands, np.nan).astype(np.float64)

    # main acquisition of the year = slot used by most bands
    cnt = np.stack([(chosen == s).sum(axis=2) for s in range(3)], axis=2)
    main = np.where(obs.any(axis=2), cnt.argmax(axis=2), -1)
    mi = np.clip(main, 0, 2)[..., None]
    has = main >= 0
    doy_main = np.where(has, np.take_along_axis(DOY, mi, axis=2)[..., 0], np.nan)
    sen_main = np.where(has, np.take_along_axis(SEN, mi, axis=2)[..., 0], -1)
    anom_main = has & np.take_along_axis(ANOM, mi, axis=2)[..., 0]
    tier2_main = has & np.take_along_axis(TIER2, mi, axis=2)[..., 0]

    # reference: what NB3 would have taken (first QA-valid slot, row dropped if Blue > 15000)
    nb3_slot = np.where(VALID.any(axis=2), VALID.argmax(axis=2), -1)
    nb3_blue = np.take_along_axis(V[..., 0], np.clip(nb3_slot, 0, 2)[..., None], axis=2)[..., 0]
    nb3_ok = (nb3_slot >= 0) & (nb3_blue <= 15_000)

    bands_observed = bands.copy() if return_internals else None   # before gap filling
    # ── gap filling (NB3 interpolation) + indices ──────────────────────────────────────
    tt = np.arange(T)
    for j in range(6):
        for i in np.nonzero(obs[..., j].any(axis=1) & ~obs[..., j].all(axis=1))[0]:
            mk = obs[i, :, j]
            bands[i, :, j] = np.interp(tt, tt[mk], bands[i, mk, j])
    n_unfillable = int(np.isnan(bands).any(axis=1).sum())
    if n_unfillable and strict:
        raise ValueError("a pixel-band without any usable observation")
    if n_unfillable:                                    # dataset's per-year median, still unobserved
        for j in range(6):
            med_y = np.nanmedian(np.where(obs[..., j], bands[..., j], np.nan), axis=0)
            bad = np.isnan(bands[..., j]).all(axis=1)
            bands[bad, :, j] = med_y[None, :]
    spec = np.concatenate([bands, np.stack(compute_indices(*(bands[..., j] for j in range(6))), axis=2)],
                          axis=2)

    meta = np.concatenate([
        obs.astype(np.float32),
        np.where(has, (doy_main - MID_SUMMER_DOY) / 60.0, 0.0)[..., None],
        np.stack([(sen_main == k) for k in range(len(SENSORS))], axis=2),
        (VALID.sum(axis=2) / 3.0)[..., None], anom_main[..., None], tier2_main[..., None]],
        axis=2).astype(np.float32)                                          # (N, T, 15)

    # ── tabular features: NB3 statistics on the v3 series + static ─────────────────────
    f = {}
    mm, sd = spec.mean(axis=1), spec.std(axis=1, ddof=1)
    for j, c in enumerate(BASE_BANDS + INDEX_COLS):
        f[f"{c}_mean"], f[f"{c}_std"] = mm[:, j], sd[:, j]
    early, late = years <= EARLY_CUTOFF, years >= LATE_START
    for tag, sel in (("early", early), ("late", late)):
        for j, b in enumerate(BASE_BANDS):
            f[f"{b}_{tag}_mean"] = bands[:, sel, j].mean(1)
            f[f"{b}_{tag}_std"] = bands[:, sel, j].std(1, ddof=1)
    dd = np.diff(bands, axis=1)
    for j, b in enumerate(BASE_BANDS):
        f[f"d_{b}_mean"], f[f"d_{b}_std"] = dd[..., j].mean(1), dd[..., j].std(1, ddof=1)
    anyobs = obs.any(axis=2)
    f["has_early_data"] = anyobs[:, early].any(1).astype(float)
    f["has_late_data"] = anyobs[:, late].any(1).astype(float)
    if centre_xy is not None:
        dist = np.hypot(px_m - centre_xy[0], py_m - centre_xy[1])
        f["coverage"], f["dist_centre_rel"] = static_cov, dist / np.percentile(dist, 95)
    else:
        dist = None
        f["coverage"], f["dist_centre_rel"] = static_cov, np.full(N, np.nan)

    if has_labels:
        b = list(class_bounds)
        age = pd.cut(pd.Series(wmy), [-np.inf] + b + [np.inf], labels=range(1, len(b) + 2)).astype(int).to_numpy()
        w, w_info = label_weights(wmy, bounds=b)
    else:
        age, w, w_info = np.full(N, -1), np.ones(N, np.float32), None
    feats = pd.DataFrame({"px_key": px_key, "py_key": py_key, "age_class": age, "label_weight": w})
    if keep_xy:                                         # for distances only -- never a model input
        feats["pixel_x"], feats["pixel_y"] = px_m, py_m
    feats = pd.concat([feats, pd.DataFrame({c: f[c] for c in TAB_FEATURES})], axis=1)

    nb3_used = np.where(nb3_ok[..., None], nb3_slot[..., None], -1)
    both = obs & (nb3_used >= 0)
    stats = dict(
        ceilings=ceil_info, saturated_by_qa_radsat=n_sat_qa, saturated_by_ceiling_only=n_sat_ceil,
        anomalous_acquisitions=anomalous_list, n_acquisitions=int(len(med)),
        observed_pixel_year_bands_v3=int(obs.sum()), observed_pixel_year_bands_nb3=int(nb3_ok.sum()) * 6,
        choice_differs_from_nb3_pct=100 * float((both & (chosen != nb3_used)).sum() / both.sum()),
        mixed_date_years=int((obs.all(2) & (chosen != chosen[..., :1]).any(2)).sum()),
        years_using_anomalous_acq=int(anom_main.sum()), years_using_tier2=int(tier2_main.sum()),
        label_weights=w_info, city_radius_km=None if dist is None else float(np.percentile(dist, 95) / 1000),
        pixel_bands_filled_from_year_median=n_unfillable, years=[int(y) for y in years])
    if return_internals:
        return feats, spec.astype(np.float32), meta, stats, dict(
            V=V, usable=usable, anomalous=ANOM, obs=obs, bands_observed=bands_observed)
    return feats, spec.astype(np.float32), meta, stats


def load_cached_v3(city="Madrid"):
    feats = pd.read_parquet(CACHE_DIR / f"{city.lower()}_features_v3.parquet")
    spec = np.load(CACHE_DIR / f"{city.lower()}_seq_spec_v3.npy", mmap_mode="r")
    meta = np.load(CACHE_DIR / f"{city.lower()}_seq_meta_v3.npz")["meta"]
    return feats, spec, meta


In [ ]:
%%writefile /kaggle/working/model_selection_final/train_final.py
"""
Final training -- TempCNN embedding, features that survived v4, Madrid only.

Inputs (per pixel)
  series (42 years x 32 channels): 6 bands + 5 indices, 6 observed masks, DOY, 5 satellite
         one-hot, 3 quality flags (v3) + 6 neighbourhood-spread channels (v4, confirmed on
         Amsterdam: +3.0 macro-F1, P = 1.00 over 20 paired seeds)
  static (2): coverage, distance to centre / city radius
Output: a 32-d L2-normalised embedding. Classes are read off by metric learning:
  nearest class prototype (primary) and k-nearest neighbours.

Evaluation: 10-fold spatial-block CV (same ~1 km blocks / splits as v3-v4): macro / per-class
F1, accuracy, confusion matrix, out-of-fold embeddings. Then a final model on all of Madrid.

Two training recipes (--recipe):
  fixed (DEFAULT, adopted)  the v3 settings that produced 0.6606 in the v4 "neigh" arm:
                  LR 1e-3, halved after 2 epochs without improvement (floor 1e-5); fixed
                  temperature 0.1; dropout 0.3; batch 256 (64 per class); grad-norm clip 1.0;
                  label-reliability weights only; at most 40 epochs, stop after 5 epochs
                  without an improvement larger than the ES score's bootstrap noise
                  (re-estimated every epoch); the checkpoint kept is the best ES epoch.
  adaptive (tested, NOT adopted)  everything below. Two Madrid 10-fold runs: 0.6505 and
                  0.6431 vs 0.6606 for the fixed recipe (paired P(better) ~0.01 both times);
                  kept for reference / reproducibility.

Adaptive recipe (nothing below is a tuned constant):
  learning rate   LR range test per fold (Smith 2017): exponential sweep, peak = LR of the
                  steepest descent of the smoothed loss (before its minimum); linear warm-up
                  over the first epoch; halved
                  whenever the early-stopping (ES) score stops improving
  stopping        an improvement counts only if larger than the ES score's own noise
                  (bootstrap std, re-estimated every epoch); training stops when the LR has
                  decayed to peak / 64 -- no fixed epoch count
  loss            supervised contrastive with a LEARNED temperature; anchor weights =
                  label reliability (heaped years / class boundaries) x dynamic class
                  weights (cumulative average of 1 - per-class ES F1, renormalised)
  dropout         follows the TREND of the train-vs-ES gap: raised only when the gap grows by
                  more than the noise, lowered when it shrinks, and a raise is undone if the
                  ES score then drops by more than the noise (run 1 used the gap's level and
                  ratcheted to the 0.5 cap: the gap is partly structural across spatial blocks)
  grad clipping   AutoClip (Seetharaman et al. 2020): 10th percentile of past grad norms
  batch size      4 classes x the power of two closest to sqrt(smallest class count)
  kNN k           sqrt(smallest class count in the training fold), odd
Fixed (validated in v1-v3): architecture size (3 conv x 64 filters, width 5, 256-unit FC,
32-d embedding), AdamW weight decay 1e-4; initial temperature 0.1 and initial dropout 0.3
are only starting points for the adaptive rules above.

Usage:
  python train_final.py --folds 0,2,4,6,8      # any subset of folds (several workers)
  python train_final.py --recipe adaptive ...  # the adaptive recipe instead of the fixed one
  python train_final.py --final                # model on all of Madrid
  python train_final.py --report               # aggregate folds -> report + confusion matrix
Env: MS_CACHE_DIR, MS_EXTRA_DIR, MS_RESULTS_DIR, MS_SMOKE=1 (tiny test)
"""
import argparse
import copy
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, silhouette_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent
for p in (HERE, HERE.parent / "model_selection_v3"):
    sys.path.insert(0, str(p))
from common_v3 import RESULTS_DIR, STATIC_COLS, TAB_FEATURES, load_cached_v3

torch.set_num_threads(int(os.environ.get("N_THREADS", "4")))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      ("mps" if torch.backends.mps.is_available() else "cpu"))
SMOKE = os.environ.get("MS_SMOKE") == "1"
EXTRA_DIR = Path(os.environ.get("MS_EXTRA_DIR", HERE.parent / "model_selection_v4" / "cache"))
OUT = RESULTS_DIR / os.environ.get("MS_RUN_NAME", "final")
N_CLASSES, N_FOLDS = 4, 10
CLASS_NAMES = ["pre-1960", "1960-1984", "1984-2004", "2004-2024"]
WEIGHT_DECAY, EMB_DIM = 1e-4, 32                       # validated in v1-v3
TAU0, DROPOUT0 = 0.1, 0.3                              # fixed values / adaptive starting points
FIXED = dict(lr=1e-3, per_class=64, lr_patience=2, lr_factor=0.5, min_lr=1e-5, grad_clip=1.0,
             max_epochs=40, patience=5)                # v3 recipe (validated v3-v4)
RECIPE = "fixed"                                       # set by --recipe


# ══════════════════════════════════════════════════════════════════════════════
# Data
# ══════════════════════════════════════════════════════════════════════════════
def load():
    feats, spec, meta = load_cached_v3("Madrid")
    neigh = np.load(EXTRA_DIR / "madrid_seq_extra_v4.npz")["neigh"].astype(np.float32)
    return feats, spec, meta, neigh


class Views:
    """Every scaler is fit on the training-fold pixels only."""

    def __init__(self, feats, spec, meta, neigh, fit_idx):
        sub = np.asarray(spec[fit_idx])
        mu = sub.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
        sd = sub.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
        nm = np.nanmean(neigh[fit_idx], axis=(0, 1)); ns = np.nanstd(neigh[fit_idx], axis=(0, 1)) + 1e-6
        seq = np.concatenate([(np.asarray(spec) - mu) / sd, np.asarray(meta, np.float32),
                              np.nan_to_num((neigh - nm) / ns, nan=0.0)], axis=2).astype(np.float32)
        tab = feats[TAB_FEATURES].to_numpy(float)
        st = [TAB_FEATURES.index(c) for c in STATIC_COLS]
        static = StandardScaler().fit(tab[fit_idx][:, st]).transform(tab[:, st]).astype(np.float32)
        self.seq = torch.from_numpy(seq).to(DEVICE)
        self.static = torch.from_numpy(static).to(DEVICE)
        self.n_channels, self.n_static = seq.shape[2], static.shape[1]

    def batch(self, idx):
        idx = torch.as_tensor(idx, device=DEVICE)
        return self.seq[idx], self.static[idx]


def folds():
    z = np.load(RESULTS_DIR / "splits_spatial10.npz")
    fid, es_bits = z["fold_id"], z["es_bits"]
    for k in range(N_FOLDS):
        va, tr = np.nonzero(fid == k)[0], np.nonzero(fid != k)[0]
        in_es = ((es_bits[tr] >> k) & 1).astype(bool)
        yield k, tr, va, tr[~in_es], tr[in_es]


def all_madrid_split(feats, y0):
    px, py = feats["px_key"].to_numpy(np.int64), feats["py_key"].to_numpy(np.int64)
    groups = ((px - px.min()) // 33) * 100_000 + (py - py.min()) // 33
    fit, es = next(StratifiedGroupKFold(7, shuffle=True, random_state=42)
                   .split(np.zeros(len(y0)), y0, groups))
    return fit, es


# ══════════════════════════════════════════════════════════════════════════════
# Model and loss
# ══════════════════════════════════════════════════════════════════════════════
class TempCNNEmb(nn.Module):
    def __init__(self, c_in, n_static, t_len=42, filters=64, kernel=5, fc=256, emb=EMB_DIM, p=DROPOUT0):
        super().__init__()
        self.drops = nn.ModuleList([nn.Dropout(p) for _ in range(5)])
        layers, c = [], c_in
        for i in range(3):
            layers += [nn.Conv1d(c, filters, kernel, padding=kernel // 2), nn.BatchNorm1d(filters),
                       nn.ReLU(), self.drops[i]]
            c = filters
        self.convs = nn.Sequential(*layers)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(filters * t_len, fc), nn.BatchNorm1d(fc),
                                nn.ReLU(), self.drops[3])
        self.head = nn.Sequential(nn.Linear(fc + n_static, 128), nn.ReLU(), self.drops[4],
                                  nn.Linear(128, emb))
        self.log_inv_tau = nn.Parameter(torch.tensor(math.log(1 / TAU0)))   # learned temperature

    def set_dropout(self, p):
        for d in self.drops:
            d.p = p

    @property
    def tau(self):
        return float(torch.exp(-self.log_inv_tau.clamp(math.log(1.0), math.log(100.0))))

    def forward(self, seq, static):
        h = self.fc(self.convs(seq.transpose(1, 2)))
        return F.normalize(self.head(torch.cat([h, static], dim=1)), dim=1)


def supcon(z, y, inv_tau, w):
    sim = z @ z.T * inv_tau
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    pos = (y[:, None] == y[None, :]) & ~eye
    log_prob = sim - torch.logsumexp(sim.masked_fill(eye, float("-inf")), dim=1, keepdim=True)
    n_pos = pos.sum(1)
    loss = -(log_prob * pos).sum(1) / n_pos.clamp_min(1)
    keep = n_pos > 0
    return (loss[keep] * w[keep]).sum() / w[keep].sum()


def balanced_batches(fit_idx, y0, per_class, rng):
    pools = [rng.permutation(fit_idx[y0[fit_idx] == c]) for c in range(N_CLASSES)]
    ptr = [0] * N_CLASSES
    for _ in range(max(1, len(fit_idx) // (per_class * N_CLASSES))):
        parts = []
        for c in range(N_CLASSES):
            if ptr[c] + per_class > len(pools[c]):
                pools[c], ptr[c] = rng.permutation(pools[c]), 0
            parts.append(pools[c][ptr[c]:ptr[c] + per_class])
            ptr[c] += per_class
        yield rng.permutation(np.concatenate(parts))


@torch.no_grad()
def embed(model, views, idx, bs=8192):
    model.eval()
    return torch.cat([model(*views.batch(idx[i:i + bs])) for i in range(0, len(idx), bs)])


def prototypes(z, y, w):
    return F.normalize(torch.stack([(z[y == c] * w[y == c, None]).sum(0) / w[y == c].sum()
                                    for c in range(N_CLASSES)]), dim=1)


def bootstrap_noise(y_true, y_pred, rng, n=50):
    """Std of macro-F1 over bootstrap resamples = how much the ES score moves by chance."""
    m = len(y_true)
    return float(np.std([f1_score(y_true[s], y_pred[s], average="macro")
                         for s in (rng.integers(0, m, m) for _ in range(n))]))


# ══════════════════════════════════════════════════════════════════════════════
# Adaptive training
# ══════════════════════════════════════════════════════════════════════════════
def lr_range_test(model, views, y_t, w_t, fit_idx, y0, per_class, rng):
    """Smith (2017): exponential LR sweep on a throw-away copy of the model."""
    m = copy.deepcopy(model)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-7, weight_decay=WEIGHT_DECAY)
    steps = 10 if SMOKE else max(60, min(300, len(fit_idx) // (per_class * N_CLASSES)))
    lrs = np.geomspace(1e-7, 1.0, steps)
    losses, smooth, best = [], None, np.inf
    batches = balanced_batches(fit_idx, y0, per_class, rng)
    for lr in lrs:
        try:
            bi = torch.as_tensor(next(batches), device=DEVICE)
        except StopIteration:
            batches = balanced_batches(fit_idx, y0, per_class, rng)
            bi = torch.as_tensor(next(batches), device=DEVICE)
        for g in opt.param_groups:
            g["lr"] = lr
        m.train(); opt.zero_grad()
        loss = supcon(m(*views.batch(bi)), y_t[bi], m.log_inv_tau.exp(), w_t[bi])
        loss.backward(); opt.step()
        smooth = loss.item() if smooth is None else 0.9 * smooth + 0.1 * loss.item()
        losses.append(smooth); best = min(best, smooth)
        if smooth > 4 * best or not np.isfinite(smooth):
            break
    L = np.array(losses); x = np.log(lrs[:len(L)])
    i_min = int(np.argmin(L))
    if i_min >= 3:
        g = np.gradient(L[:i_min + 1], x[:i_min + 1])
        i_steep = int(np.argmin(g[1:])) + 1             # steepest descent before the minimum
        lr = float(lrs[i_steep])
    else:
        lr = float(lrs[i_min] / 10)
    return lr, dict(lrs=lrs[:len(L)].tolist(), losses=losses, i_min=i_min,
                    rule='steepest descent' if i_min >= 3 else 'min/10 fallback')


def train(views, y0, w0, fit_idx, es_idx, seed, log=print):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    y_t, w_label = torch.as_tensor(y0, device=DEVICE), torch.as_tensor(w0, device=DEVICE)
    n_min = int(np.bincount(y0[fit_idx], minlength=N_CLASSES).min())
    per_class = 2 ** int(round(math.log2(math.sqrt(n_min))))            # batch = 4 x per_class
    if SMOKE:
        per_class = min(per_class, 32)
    model = TempCNNEmb(views.n_channels, views.n_static).to(DEVICE)
    class_w = torch.ones(N_CLASSES, device=DEVICE)
    lr_peak, lr_curve = lr_range_test(model, views, y_t, w_label * class_w[y_t], fit_idx, y0, per_class, rng)
    opt = torch.optim.AdamW(model.parameters(), lr=lr_peak, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = max(1, len(fit_idx) // (per_class * N_CLASSES))
    warmup = steps_per_epoch                                               # one epoch of warm-up
    probe = rng.choice(fit_idx, min(len(es_idx), len(fit_idx)), replace=False)   # train-side probe
    probe_t = torch.as_tensor(probe, device=DEVICE)
    es_t = torch.as_tensor(es_idx, device=DEVICE)
    grad_hist, step, lr = [], 0, lr_peak
    best, best_state, best_epoch, wait, cw_sum, epoch = -1.0, None, 0, 0, torch.zeros(N_CLASSES, device=DEVICE), 0
    history, prev_gap, raised = [], None, None
    max_epochs = 2 if SMOKE else 500                                       # safety only; LR floor stops
    t0 = time.time()
    while epoch < max_epochs:
        epoch += 1
        model.train()
        tot, nb = 0.0, 0
        for bi in balanced_batches(fit_idx, y0, per_class, rng):
            bi = torch.as_tensor(bi, device=DEVICE)
            step += 1
            for g in opt.param_groups:
                g["lr"] = lr * min(1.0, step / warmup)
            opt.zero_grad()
            loss = supcon(model(*views.batch(bi)), y_t[bi], model.log_inv_tau.exp().clamp(1.0, 100.0),
                          w_label[bi] * class_w[y_t[bi]])
            loss.backward()
            gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf")))
            if math.isfinite(gn):
                grad_hist.append(gn)
            clip = float(np.percentile(grad_hist, 10)) if len(grad_hist) >= 20 else float("inf")
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)               # AutoClip
            opt.step()
            tot, nb = tot + loss.item(), nb + 1
        # ── epoch-level adaptation ────────────────────────────────────────────
        P = prototypes(embed(model, views, fit_idx), y_t[torch.as_tensor(fit_idx, device=DEVICE)],
                       w_label[torch.as_tensor(fit_idx, device=DEVICE)])
        pred_es = (embed(model, views, es_idx) @ P.T).argmax(1).cpu().numpy()
        pred_tr = (embed(model, views, probe) @ P.T).argmax(1).cpu().numpy()
        es_f1 = float(f1_score(y0[es_idx], pred_es, average="macro"))
        tr_f1 = float(f1_score(y0[probe], pred_tr, average="macro"))
        noise = bootstrap_noise(y0[es_idx], pred_es, rng)
        per_c = f1_score(y0[es_idx], pred_es, average=None, labels=range(N_CLASSES))
        hard = torch.as_tensor(1 - per_c, dtype=torch.float32, device=DEVICE) + 1e-3
        cw_sum += hard / hard.mean()
        class_w = cw_sum / epoch                                             # cumulative average
        gap = tr_f1 - es_f1
        p = model.drops[0].p
        if raised is not None and es_f1 < raised[1] - noise:
            p, raised = raised[0], None                      # the last raise hurt: undo it
        elif prev_gap is not None and gap - prev_gap > noise:
            raised, p = (p, es_f1), min(0.5, p + 0.05)       # gap growing: regularise more
        elif prev_gap is not None and gap - prev_gap < -noise:
            raised, p = None, max(0.0, p - 0.05)             # gap shrinking: relax
        else:
            raised = None
        prev_gap = gap
        model.set_dropout(p)
        improved = es_f1 > best + noise
        if improved:
            best, best_epoch, wait, best_state = es_f1, epoch, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= 2:                                                    # plateau -> halve
                lr, wait = lr / 2, 0
        history.append(dict(epoch=epoch, loss=tot / nb, es_f1=es_f1, train_probe_f1=tr_f1, noise=noise,
                            lr=lr, tau=model.tau, dropout=p, class_w=class_w.tolist(),
                            clip=clip if math.isfinite(clip) else None, seconds=time.time() - t0))
        log(f"    ep {epoch:3d} loss {tot / nb:.3f} | ES F1 {es_f1:.4f} (noise {noise:.4f}) train {tr_f1:.4f} | "
            f"lr {lr:.2e} tau {model.tau:.3f} drop {p:.2f} | class_w {np.round(class_w.tolist(), 2).tolist()}")
        if lr < lr_peak / 64:
            break
    model.load_state_dict(best_state)
    info = dict(lr_peak=lr_peak, batch_size=per_class * N_CLASSES, best_epoch=best_epoch, epochs_run=epoch,
                best_es_f1=best, final_tau=history[best_epoch - 1]["tau"], seconds=time.time() - t0)
    return model, info, history, lr_curve

def train_fixed(views, y0, w0, fit_idx, es_idx, seed, log=print, keep_states=None):
    """The v3 recipe; the only data-driven part is the noise threshold for 'improvement'.
    keep_states: optional list; receives (es_f1, noise, cpu state_dict) for every epoch (for SWA)."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    y_t, w_t = torch.as_tensor(y0, device=DEVICE), torch.as_tensor(w0, device=DEVICE)
    per_class = min(FIXED["per_class"], 32) if SMOKE else FIXED["per_class"]
    model = TempCNNEmb(views.n_channels, views.n_static, p=DROPOUT0).to(DEVICE)
    model.log_inv_tau.requires_grad_(False)                               # temperature fixed at 0.1
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=FIXED["lr"],
                            weight_decay=WEIGHT_DECAY)
    fit_t = torch.as_tensor(fit_idx, device=DEVICE)
    lr, best, best_sig, best_state, best_epoch, wait, lr_wait = FIXED["lr"], -1.0, -1.0, None, 0, 0, 0
    history, t0 = [], time.time()
    for epoch in range(1, (2 if SMOKE else FIXED["max_epochs"]) + 1):
        model.train()
        tot, nb = 0.0, 0
        for bi in balanced_batches(fit_idx, y0, per_class, rng):
            bi = torch.as_tensor(bi, device=DEVICE)
            opt.zero_grad()
            loss = supcon(model(*views.batch(bi)), y_t[bi], 1.0 / TAU0, w_t[bi])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), FIXED["grad_clip"])
            opt.step()
            tot, nb = tot + loss.item(), nb + 1
        P = prototypes(embed(model, views, fit_idx), y_t[fit_t], w_t[fit_t])
        pred_es = (embed(model, views, es_idx) @ P.T).argmax(1).cpu().numpy()
        es_f1 = float(f1_score(y0[es_idx], pred_es, average="macro"))
        noise = bootstrap_noise(y0[es_idx], pred_es, rng)
        if keep_states is not None:
            keep_states.append((es_f1, noise, {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}))
        if es_f1 > best:                                                   # checkpoint = best ES epoch
            best, best_epoch, best_state = es_f1, epoch, copy.deepcopy(model.state_dict())
        if es_f1 > best_sig + noise:                                       # a real improvement
            best_sig, wait, lr_wait = es_f1, 0, 0
        else:
            wait, lr_wait = wait + 1, lr_wait + 1
            if lr_wait > FIXED["lr_patience"] and lr > FIXED["min_lr"]:
                lr, lr_wait = max(FIXED["min_lr"], lr * FIXED["lr_factor"]), 0
                for g in opt.param_groups:
                    g["lr"] = lr
        history.append(dict(epoch=epoch, loss=tot / nb, es_f1=es_f1, noise=noise, lr=lr, tau=TAU0,
                            dropout=DROPOUT0, class_w=[1.0] * N_CLASSES, seconds=time.time() - t0))
        log(f"    ep {epoch:3d} loss {tot / nb:.3f} | ES F1 {es_f1:.4f} (noise {noise:.4f}) | lr {lr:.1e}")
        if wait >= FIXED["patience"]:
            break
    model.load_state_dict(best_state)
    info = dict(recipe="fixed", lr_peak=FIXED["lr"], batch_size=per_class * N_CLASSES, best_epoch=best_epoch,
                epochs_run=epoch, best_es_f1=best, final_tau=TAU0, seconds=time.time() - t0)
    return model, info, history, None


@torch.no_grad()
def evaluate(model, views, y0, w0, fit_idx, va_idx):
    y_t, w_t = torch.as_tensor(y0, device=DEVICE), torch.as_tensor(w0, device=DEVICE)
    fit_t, va_t = torch.as_tensor(fit_idx, device=DEVICE), torch.as_tensor(va_idx, device=DEVICE)
    z_fit, z_va = embed(model, views, fit_idx), embed(model, views, va_idx)
    P = prototypes(z_fit, y_t[fit_t], w_t[fit_t])
    pred = (z_va @ P.T).argmax(1).cpu().numpy()
    n_min = int(np.bincount(y0[fit_idx], minlength=N_CLASSES).min())
    k = int(math.sqrt(n_min)) | 1
    sims = torch.cat([(z_va[i:i + 2048] @ z_fit.T) for i in range(0, len(va_idx), 2048)])
    top = sims.topk(k, dim=1).indices
    votes = torch.zeros(len(va_idx), N_CLASSES, device=DEVICE).scatter_add_(
        1, y_t[fit_t][top], torch.ones_like(top, dtype=torch.float32))
    knn = votes.argmax(1).cpu().numpy()
    yv = y0[va_idx]
    rng = np.random.default_rng(0)
    sub = np.concatenate([rng.choice(np.nonzero(yv == c)[0], min(500, int((yv == c).sum())), replace=False)
                          for c in range(N_CLASSES)])
    return dict(proto_macro_f1=float(f1_score(yv, pred, average="macro")),
                proto_per_class_f1=f1_score(yv, pred, average=None, labels=range(N_CLASSES)).tolist(),
                proto_accuracy=float(accuracy_score(yv, pred)),
                knn_k=k, knn_macro_f1=float(f1_score(yv, knn, average="macro")),
                confusion=confusion_matrix(yv, pred, labels=range(N_CLASSES)).tolist(),
                silhouette_cosine=float(silhouette_score(z_va[torch.as_tensor(sub, device=DEVICE)].cpu().numpy(),
                                                         yv[sub], metric="cosine")),
                prototypes=P.cpu().numpy().tolist()), z_va.cpu().numpy(), pred


# ══════════════════════════════════════════════════════════════════════════════
def run_fold(k, data, log):
    feats, spec, meta, neigh = data
    y0, w0 = feats["age_class"].to_numpy() - 1, feats["label_weight"].to_numpy(np.float32)
    _, tr, va, fit, es = [f for f in folds() if f[0] == k][0]
    if SMOKE:
        r = np.random.default_rng(k)
        fit, es, va = r.choice(fit, 2048, False), r.choice(es, 1024, False), r.choice(va, 2048, False)
    views = Views(feats, spec, meta, neigh, tr)
    log(f"\n=== fold {k} ({DEVICE}, recipe {RECIPE}): fit {len(fit):,} es {len(es):,} val {len(va):,}, "
        f"{views.n_channels} channels + {views.n_static} static ===")
    if RECIPE == "fixed":
        model, info, hist, lr_curve = train_fixed(views, y0, w0, fit, es, seed=1000 + k, log=log)
    else:
        model, info, hist, lr_curve = train(views, y0, w0, fit, es, seed=100 + k, log=log)
    m, z_va, pred = evaluate(model, views, y0, w0, fit, va)
    OUT.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(OUT / f"fold{k}_oof.npz", idx=va, z=z_va.astype(np.float16), pred=pred)
    (OUT / f"fold{k}.json").write_text(json.dumps(dict(fold=k, info=info, val=m, history=hist,
                                                       lr_range_test=lr_curve), indent=1))
    log(f"  -> fold {k}: proto macro-F1 {m['proto_macro_f1']:.4f} | kNN-F1 {m['knn_macro_f1']:.4f} "
        f"(k={m['knn_k']}) | lr_peak {info['lr_peak']:.2e} | batch {info['batch_size']} | "
        f"best ep {info['best_epoch']}/{info['epochs_run']} | {info['seconds']:.0f}s")


def run_final(data, log):
    feats, spec, meta, neigh = data
    y0, w0 = feats["age_class"].to_numpy() - 1, feats["label_weight"].to_numpy(np.float32)
    fit, es = all_madrid_split(feats, y0)
    if SMOKE:
        r = np.random.default_rng(0)
        fit, es = r.choice(fit, 2048, False), r.choice(es, 1024, False)
    views = Views(feats, spec, meta, neigh, np.arange(len(y0)))
    log(f"\n=== FINAL model on all Madrid ({DEVICE}): fit {len(fit):,} es {len(es):,} ===")
    trainer = train_fixed if RECIPE == "fixed" else train
    model, info, hist, lr_curve = trainer(views, y0, w0, fit, es, seed=999, log=log)
    allidx = np.arange(len(y0))
    z = embed(model, views, allidx)
    P = prototypes(z, torch.as_tensor(y0, device=DEVICE), torch.as_tensor(w0, device=DEVICE))
    OUT.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "n_channels": views.n_channels, "n_static": views.n_static,
                "prototypes": P.cpu(), "info": info}, OUT / "final_model.pt")
    np.savez_compressed(OUT / "final_embeddings.npz", z=z.cpu().numpy().astype(np.float16),
                        px_key=feats["px_key"].to_numpy(), py_key=feats["py_key"].to_numpy(),
                        age_class=feats["age_class"].to_numpy(), prototypes=P.cpu().numpy())
    (OUT / "final.json").write_text(json.dumps(dict(info=info, history=hist, lr_range_test=lr_curve), indent=1))
    log(f"  -> final model: best ep {info['best_epoch']}/{info['epochs_run']}, ES F1 {info['best_es_f1']:.4f}, "
        f"{info['seconds']:.0f}s")


def report():
    J = [json.loads((OUT / f"fold{k}.json").read_text()) for k in range(N_FOLDS) if (OUT / f"fold{k}.json").exists()]
    f1 = np.array([j["val"]["proto_macro_f1"] for j in J])
    knn = np.array([j["val"]["knn_macro_f1"] for j in J])
    pc = np.array([j["val"]["proto_per_class_f1"] for j in J])
    cm = np.sum([np.array(j["val"]["confusion"]) for j in J], axis=0)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    L = ["# Final model — TempCNN embedding, Madrid 10-fold spatial-block CV\n",
         f"{len(J)}/{N_FOLDS} folds. Inputs: 42 years x 32 channels + 2 static. Metric-learning evaluation.\n",
         "| Metric | Mean ± sd over folds |", "|---|---|",
         f"| Prototype macro-F1 | **{f1.mean():.4f}** ± {f1.std(ddof=1) if len(f1) > 1 else 0:.4f} |",
         f"| kNN macro-F1 | {knn.mean():.4f} ± {knn.std(ddof=1) if len(knn) > 1 else 0:.4f} |",
         f"| Accuracy | {np.mean([j['val']['proto_accuracy'] for j in J]):.4f} |",
         f"| Silhouette (cosine) | {np.mean([j['val']['silhouette_cosine'] for j in J]):.3f} |"]
    L += ["\n## Per-class F1\n", "| " + " | ".join(CLASS_NAMES) + " |", "|" + "---:|" * 4,
          "| " + " | ".join(f"{x:.3f}" for x in pc.mean(0)) + " |"]
    L += ["\n## Confusion matrix (all folds, rows = true, normalised per row)\n",
          "| true \\ predicted | " + " | ".join(CLASS_NAMES) + " | n |", "|---|" + "---:|" * 5]
    for i, n in enumerate(CLASS_NAMES):
        L.append(f"| **{n}** | " + " | ".join(f"{x:.2f}" for x in cmn[i]) + f" | {cm[i].sum():,} |")
    recipe = J[0]["info"].get("recipe", "adaptive") if J else "?"
    L.insert(2, f"Training recipe: **{recipe}**.\n")
    L += [f"\n## Training per fold ({recipe} recipe)\n",
          "| Fold | Peak LR | Batch | Best / run epochs | Final temperature | Final dropout | Final class weights |",
          "|---:|---:|---:|---|---:|---:|---|"]
    for j in J:
        h = j["history"][j["info"]["best_epoch"] - 1]
        L.append(f"| {j['fold']} | {j['info']['lr_peak']:.2e} | {j['info']['batch_size']} | "
                 f"{j['info']['best_epoch']} / {j['info']['epochs_run']} | {h['tau']:.3f} | {h['dropout']:.2f} | "
                 f"{np.round(h['class_w'], 2).tolist()} |")
    (OUT / "final_report.md").write_text("\n".join(L) + "\n")
    np.savetxt(OUT / "confusion_counts.csv", cm, fmt="%d", delimiter=",", header=",".join(CLASS_NAMES))
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5.2, 4.4))
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(4), CLASS_NAMES, rotation=30, ha="right"); ax.set_yticks(range(4), CLASS_NAMES)
        for i in range(4):
            for jj in range(4):
                ax.text(jj, i, f"{cmn[i, jj]:.2f}", ha="center", va="center",
                        color="white" if cmn[i, jj] > 0.5 else "black", fontsize=9)
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
        ax.set_title(f"Madrid 10-fold CV — macro-F1 {f1.mean():.3f}")
        fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout(); fig.savefig(OUT / "confusion_matrix.png", dpi=150)
    except ImportError:
        pass
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--folds", default="")
    ap.add_argument("--final", action="store_true")
    ap.add_argument("--report", action="store_true")
    ap.add_argument("--recipe", choices=["fixed", "adaptive"], default="fixed")
    a = ap.parse_args()
    global RECIPE
    RECIPE = a.recipe
    log = lambda s: print(s, flush=True)
    if a.folds or a.final:
        data = load()
        log(f"device={DEVICE} smoke={SMOKE} recipe={RECIPE}")
        for k in [int(x) for x in a.folds.split(",") if x.strip()]:
            run_fold(k, data, log)
        if a.final:
            run_final(data, log)
        log("WORKER DONE")
    if a.report:
        report()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/model_selection_final/ensemble_final.py
"""
Backbone for transfer: seed ensemble + weight averaging (SWA) of the fixed-recipe TempCNN.
Madrid only (training and evaluation).

Per model (fixed recipe, train_final.train_fixed), two variants from the SAME run:
  best  the best early-stopping (ES) epoch (as in train_final.py)
  swa   average of the weights of every epoch whose ES F1 is within the ES noise of the
        best epoch (the plateau; width chosen by the data), BatchNorm statistics
        re-estimated on the training pixels afterwards (Izmailov et al. 2018)
Ensemble of M models: class scores = mean over models of cos(z_m, prototype_m), i.e. the
ensemble embedding is the concatenation of the M unit embeddings / sqrt(M)  (32*M dims).

CV (Madrid, same 10 spatial folds): N_SEEDS models per fold -> macro-F1 for M = 1..N_SEEDS,
for both variants, prototype and kNN. Decision rule (fixed before seeing results): the
variant used for the backbone is `swa` only if its full-ensemble CV macro-F1 beats `best`
by more than the ROPE (0.005) with P >= 0.95 (Bayesian correlated t-test); else `best`.

Final: N_SEEDS models on all Madrid (fit / ES split as train_final.py), saved together with
the normalisation statistics needed to embed any other city -> backbone.pt, plus the
Madrid ensemble embeddings of every pixel.

Usage:
  python ensemble_final.py --folds 0,2,4,6,8     # CV workers
  python ensemble_final.py --final               # backbone on all Madrid
  python ensemble_final.py --report
Env: as train_final.py, plus MS_N_SEEDS (default 10)
"""
import argparse
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy import stats as sstats
from sklearn.metrics import confusion_matrix, f1_score

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import train_final as TF
from train_final import (CLASS_NAMES, DEVICE, DROPOUT0, N_CLASSES, N_FOLDS, RESULTS_DIR, SMOKE,
                         STATIC_COLS, TAB_FEATURES, TempCNNEmb, Views, all_madrid_split, embed, folds, load,
                         prototypes, train_fixed)

N_SEEDS = 2 if SMOKE else int(os.environ.get("MS_N_SEEDS", "10"))
OUT = RESULTS_DIR / os.environ.get("MS_RUN_NAME", "ensemble")
VARIANTS = ("best", "swa")
ROPE = 0.005


# ══════════════════════════════════════════════════════════════════════════════
# Normalisation that can be applied to any city (stats from Madrid training pixels)
# ══════════════════════════════════════════════════════════════════════════════
def norm_stats(feats, spec, neigh, fit_idx):
    sub = np.asarray(spec[fit_idx])
    tab = feats[TAB_FEATURES].to_numpy(float)[fit_idx][:, [TAB_FEATURES.index(c) for c in STATIC_COLS]]
    return dict(spec_mu=sub.mean(axis=(0, 1), dtype=np.float64).astype(np.float32),
                spec_sd=sub.std(axis=(0, 1), dtype=np.float64).astype(np.float32),
                neigh_mu=np.nanmean(neigh[fit_idx], axis=(0, 1)).astype(np.float32),
                neigh_sd=(np.nanstd(neigh[fit_idx], axis=(0, 1)) + 1e-6).astype(np.float32),
                static_mu=tab.mean(0).astype(np.float32), static_sd=tab.std(0).astype(np.float32),
                static_cols=list(STATIC_COLS))


def make_views(feats, spec, meta, neigh, st):
    """Same transform as train_final.Views, with given statistics."""
    seq = np.concatenate([(np.asarray(spec) - st["spec_mu"]) / st["spec_sd"], np.asarray(meta, np.float32),
                          np.nan_to_num((neigh - st["neigh_mu"]) / st["neigh_sd"], nan=0.0)], axis=2)
    tab = feats[TAB_FEATURES].to_numpy(float)[:, [TAB_FEATURES.index(c) for c in STATIC_COLS]]
    v = Views.__new__(Views)
    v.seq = torch.from_numpy(seq.astype(np.float32)).to(DEVICE)
    v.static = torch.from_numpy(((tab - st["static_mu"]) / st["static_sd"]).astype(np.float32)).to(DEVICE)
    v.n_channels, v.n_static = seq.shape[2], tab.shape[1]
    return v


# ══════════════════════════════════════════════════════════════════════════════
# SWA over the ES plateau
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def bn_update(model, views, idx, bs=4096):
    bns = [m for m in model.modules() if isinstance(m, nn.BatchNorm1d)]
    for m in bns:
        m.reset_running_stats(); m.momentum = None          # cumulative average over batches
    model.train(); model.set_dropout(0.0)
    perm = np.random.default_rng(0).permutation(idx)
    for i in range(0, len(perm), bs):
        model(*views.batch(perm[i:i + bs]))
    for m in bns:
        m.momentum = 0.1
    model.set_dropout(DROPOUT0); model.eval()


def swa_model(states, views, fit_idx):
    f_best, noise_best, _ = max(states, key=lambda e: e[0])
    sel = [st for f, _, st in states if f >= f_best - noise_best]
    avg = {n: (torch.stack([s[n].float() for s in sel]).mean(0).to(t.dtype) if t.is_floating_point() else sel[-1][n])
           for n, t in sel[0].items()}
    m = TempCNNEmb(views.n_channels, views.n_static).to(DEVICE)
    m.load_state_dict({n: t.to(DEVICE) for n, t in avg.items()})
    bn_update(m, views, fit_idx)
    return m, len(sel)


# ══════════════════════════════════════════════════════════════════════════════
# Ensemble scoring
# ══════════════════════════════════════════════════════════════════════════════
def ens_proto_scores(zs, Ps):
    """mean over models of cos(z_m, P_m)."""
    return sum(z.to(DEVICE).float() @ P.to(DEVICE).T for z, P in zip(zs, Ps)) / len(zs)


def ens_knn(zq_list, zr_list, y_ref, k, chunk=2048):
    M = len(zq_list)
    zq = torch.cat([z.to(DEVICE).float() for z in zq_list], 1) / math.sqrt(M)
    zr = torch.cat([z.to(DEVICE).float() for z in zr_list], 1) / math.sqrt(M)
    yr = torch.as_tensor(y_ref, device=DEVICE)
    out = []
    for i in range(0, len(zq), chunk):
        top = (zq[i:i + chunk] @ zr.T).topk(k, dim=1).indices
        votes = torch.zeros(len(top), N_CLASSES, device=DEVICE).scatter_add_(
            1, yr[top], torch.ones_like(top, dtype=torch.float32))
        out.append(votes.argmax(1))
    return torch.cat(out).cpu().numpy()


def mf1(y, p):
    return float(f1_score(y, p, average="macro"))


# ══════════════════════════════════════════════════════════════════════════════
def train_pair(views, y0, w0, fit, es, seed, log):
    states = []
    model, info, hist, _ = train_fixed(views, y0, w0, fit, es, seed=seed, log=lambda s: None, keep_states=states)
    swa, n_avg = swa_model(states, views, fit)
    del states
    info["swa_epochs_averaged"] = n_avg
    return {"best": model, "swa": swa}, info


def run_fold(k, data, log):
    feats, spec, meta, neigh = data
    y0, w0 = feats["age_class"].to_numpy() - 1, feats["label_weight"].to_numpy(np.float32)
    _, tr, va, fit, es = [f for f in folds() if f[0] == k][0]
    if SMOKE:
        r = np.random.default_rng(k)
        fit, es, va = r.choice(fit, 2048, False), r.choice(es, 1024, False), r.choice(va, 2048, False)
    views = Views(feats, spec, meta, neigh, tr)
    y_t, w_t = torch.as_tensor(y0, device=DEVICE), torch.as_tensor(w0, device=DEVICE)
    fit_t = torch.as_tensor(fit, device=DEVICE)
    yv = y0[va]
    log(f"\n=== fold {k} ({DEVICE}): {N_SEEDS} seeds, fit {len(fit):,} es {len(es):,} val {len(va):,} ===")
    Z = {v: dict(zf=[], zv=[], P=[]) for v in VARIANTS}
    singles, infos, t0 = {v: [] for v in VARIANTS}, [], time.time()
    for s in range(N_SEEDS):
        models, info = train_pair(views, y0, w0, fit, es, seed=1000 + k + 100 * s, log=log)
        infos.append(info)
        msg = []
        for v, m in models.items():
            zf, zv = embed(m, views, fit), embed(m, views, va)
            P = prototypes(zf, y_t[fit_t], w_t[fit_t])
            Z[v]["zf"].append(zf.half().cpu()); Z[v]["zv"].append(zv.cpu()); Z[v]["P"].append(P.cpu())
            singles[v].append(mf1(yv, (zv @ P.T).argmax(1).cpu().numpy()))
            msg.append(f"{v} {singles[v][-1]:.4f}")
        log(f"  seed {s}: " + " | ".join(msg) + f" | best ep {info['best_epoch']}/{info['epochs_run']}, "
            f"swa avg {info['swa_epochs_averaged']} ep | {time.time() - t0:.0f}s")
        del models
    n_min = int(np.bincount(y0[fit], minlength=N_CLASSES).min())
    kk = int(math.sqrt(n_min)) | 1
    res = dict(fold=k, n_seeds=N_SEEDS, knn_k=kk, single_f1=singles, infos=infos, curve={}, knn={}, per_class={},
               confusion={}, accuracy={})
    for v in VARIANTS:
        res["curve"][v] = []
        for M in range(1, N_SEEDS + 1):
            pred = ens_proto_scores(Z[v]["zv"][:M], Z[v]["P"][:M]).argmax(1).cpu().numpy()
            res["curve"][v].append(mf1(yv, pred))
        res["per_class"][v] = f1_score(yv, pred, average=None, labels=range(N_CLASSES)).tolist()
        res["confusion"][v] = confusion_matrix(yv, pred, labels=range(N_CLASSES)).tolist()
        res["accuracy"][v] = float((pred == yv).mean())
        res["knn"][v] = {str(M): mf1(yv, ens_knn(Z[v]["zv"][:M], Z[v]["zf"][:M], y0[fit], kk))
                         for M in sorted({1, N_SEEDS})}
        zens = torch.cat(Z[v]["zv"], 1) / math.sqrt(N_SEEDS)
        np.savez_compressed(OUT / f"fold{k}_{v}_oof.npz", idx=va, z=zens.numpy().astype(np.float16), pred=pred)
    (OUT / f"fold{k}.json").write_text(json.dumps(res, indent=1))
    log(f"  -> fold {k}: single best {np.mean(singles['best']):.4f} swa {np.mean(singles['swa']):.4f} | "
        f"ensemble({N_SEEDS}) best {res['curve']['best'][-1]:.4f} swa {res['curve']['swa'][-1]:.4f} | "
        f"{time.time() - t0:.0f}s")


def run_final(data, log):
    feats, spec, meta, neigh = data
    y0, w0 = feats["age_class"].to_numpy() - 1, feats["label_weight"].to_numpy(np.float32)
    fit, es = all_madrid_split(feats, y0)
    if SMOKE:
        r = np.random.default_rng(0)
        fit, es = r.choice(fit, 2048, False), r.choice(es, 1024, False)
    allm = np.arange(len(y0))
    st = norm_stats(feats, spec, neigh, allm)
    vm = make_views(feats, spec, meta, neigh, st)
    y_t, w_t = torch.as_tensor(y0, device=DEVICE), torch.as_tensor(w0, device=DEVICE)
    log(f"\n=== FINAL backbone on all Madrid ({DEVICE}): {N_SEEDS} seeds ===")
    keep = {v: dict(state=[], zm=[], P=[]) for v in VARIANTS}
    infos, t0 = [], time.time()
    for s in range(N_SEEDS):
        models, info = train_pair(vm, y0, w0, fit, es, seed=999 + 100 * s, log=log)
        infos.append(info)
        for v, m in models.items():
            zm = embed(m, vm, allm)
            keep[v]["state"].append({n: t.cpu() for n, t in m.state_dict().items()})
            keep[v]["zm"].append(zm.half().cpu())
            keep[v]["P"].append(prototypes(zm, y_t, w_t).cpu())
        log(f"  seed {s}: best ep {info['best_epoch']}/{info['epochs_run']}, ES F1 {info['best_es_f1']:.4f}, "
            f"swa avg {info['swa_epochs_averaged']} ep | {time.time() - t0:.0f}s")
        del models
    OUT.mkdir(parents=True, exist_ok=True)
    torch.save(dict(states={v: keep[v]["state"] for v in VARIANTS},
                    madrid_prototypes={v: torch.stack(keep[v]["P"]) for v in VARIANTS},
                    norm=st, n_channels=vm.n_channels, n_static=vm.n_static, emb_dim=TF.EMB_DIM,
                    arch=dict(filters=64, kernel=5, fc=256, t_len=42), class_names=CLASS_NAMES,
                    channels="v3 series (11 spectral, 6 obs masks, doy, 5 sensor, 3 quality) + 6 neigh std",
                    infos=infos), OUT / "backbone.pt")
    for v in VARIANTS:
        np.savez_compressed(OUT / f"backbone_{v}_embeddings.npz",
                            madrid_z=(torch.cat(keep[v]["zm"], 1).float() / math.sqrt(N_SEEDS)).numpy().astype(np.float16),
                            madrid_px=feats["px_key"].to_numpy(), madrid_py=feats["py_key"].to_numpy(),
                            madrid_age_class=feats["age_class"].to_numpy())
    log(f"  -> final backbone saved ({time.time() - t0:.0f}s)")


# ══════════════════════════════════════════════════════════════════════════════
def corr_t(d, k=N_FOLDS):
    rho = 1 / k
    sd = math.sqrt((1 / len(d) + rho / (1 - rho)) * np.var(d, ddof=1)) if len(d) > 1 else 1e-9
    return sstats.t(df=len(d) - 1, loc=float(np.mean(d)), scale=max(sd, 1e-9))


def report():
    J = [json.loads((OUT / f"fold{k}.json").read_text()) for k in range(N_FOLDS) if (OUT / f"fold{k}.json").exists()]
    M = J[0]["n_seeds"]
    cur = {v: np.array([j["curve"][v] for j in J]) for v in VARIANTS}            # (folds, M)
    single = {v: np.array([np.mean(j["single_f1"][v]) for j in J]) for v in VARIANTS}
    ens = {v: cur[v][:, -1] for v in VARIANTS}
    d = ens["swa"] - ens["best"]
    t = corr_t(d)
    choice = "swa" if (1 - t.cdf(ROPE)) >= 0.95 else "best"
    L = ["# Backbone — seed ensemble + SWA, TempCNN fixed recipe, Madrid 10-fold spatial CV\n",
         f"{len(J)}/{N_FOLDS} folds, {M} seeds per fold. Prototype macro-F1 unless stated.\n",
         "| | best checkpoint | SWA (ES plateau) |", "|---|---:|---:|",
         f"| Single model (mean of {M} seeds) | {single['best'].mean():.4f} | {single['swa'].mean():.4f} |",
         f"| Ensemble of {M} | **{ens['best'].mean():.4f}** ± {ens['best'].std(ddof=1):.4f} | "
         f"**{ens['swa'].mean():.4f}** ± {ens['swa'].std(ddof=1):.4f} |",
         f"| kNN, single (seed 0) | {np.mean([j['knn'][v]['1'] for j in J for v in ['best']]):.4f} | "
         f"{np.mean([j['knn']['swa']['1'] for j in J]):.4f} |",
         f"| kNN, ensemble of {M} | {np.mean([j['knn']['best'][str(M)] for j in J]):.4f} | "
         f"{np.mean([j['knn']['swa'][str(M)] for j in J]):.4f} |",
         f"| Accuracy, ensemble | {np.mean([j['accuracy']['best'] for j in J]):.4f} | "
         f"{np.mean([j['accuracy']['swa'] for j in J]):.4f} |",
         f"\nSWA − best (ensembles): {d.mean():+.4f}, 95% [{t.ppf(.025):+.4f}, {t.ppf(.975):+.4f}], "
         f"P(SWA better by > ROPE {ROPE}) = {1 - t.cdf(ROPE):.3f} → **backbone variant: {choice}** "
         f"(pre-registered rule).\n"]
    for v in VARIANTS:
        de = ens[v] - single[v]
        te = corr_t(de)
        L.append(f"Ensemble vs single ({v}): {de.mean():+.4f}, 95% [{te.ppf(.025):+.4f}, {te.ppf(.975):+.4f}], "
                 f"wins {int((de > 0).sum())}/{len(de)}.\n")
    L += ["\n## Ensemble size (mean over folds)\n", "| M | " + " | ".join(map(str, range(1, M + 1))) + " |",
          "|---|" + "---:|" * M]
    for v in VARIANTS:
        L.append(f"| {v} | " + " | ".join(f"{x:.4f}" for x in cur[v].mean(0)) + " |")
    pc = np.mean([j["per_class"][choice] for j in J], axis=0)
    cm = np.sum([np.array(j["confusion"][choice]) for j in J], axis=0)
    cmn = cm / cm.sum(1, keepdims=True)
    L += [f"\n## Per-class F1 (ensemble, {choice})\n", "| " + " | ".join(CLASS_NAMES) + " |", "|" + "---:|" * 4,
          "| " + " | ".join(f"{x:.3f}" for x in pc) + " |",
          f"\n## Confusion matrix (ensemble, {choice}; rows = true, normalised)\n",
          "| true \\ predicted | " + " | ".join(CLASS_NAMES) + " | n |", "|---|" + "---:|" * 5]
    for i, n in enumerate(CLASS_NAMES):
        L.append(f"| **{n}** | " + " | ".join(f"{x:.2f}" for x in cmn[i]) + f" | {cm[i].sum():,} |")
    L += ["\n## Per fold (ensemble)\n", "| Fold | best | SWA | mean SWA epochs averaged |", "|---:|---:|---:|---:|"]
    for j in J:
        L.append(f"| {j['fold']} | {j['curve']['best'][-1]:.4f} | {j['curve']['swa'][-1]:.4f} | "
                 f"{np.mean([i['swa_epochs_averaged'] for i in j['infos']]):.1f} |")
    np.savetxt(OUT / "confusion_counts.csv", cm, fmt="%d", delimiter=",", header=",".join(CLASS_NAMES))
    (OUT / "ensemble_report.md").write_text("\n".join(L) + "\n")
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
        im = ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax[0].set_xticks(range(4), CLASS_NAMES, rotation=30, ha="right"); ax[0].set_yticks(range(4), CLASS_NAMES)
        for i in range(4):
            for jj in range(4):
                ax[0].text(jj, i, f"{cmn[i, jj]:.2f}", ha="center", va="center",
                           color="white" if cmn[i, jj] > 0.5 else "black", fontsize=9)
        ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
        ax[0].set_title(f"Ensemble ({choice}) — macro-F1 {ens[choice].mean():.3f}")
        fig.colorbar(im, ax=ax[0], fraction=0.046)
        for v, c in zip(VARIANTS, ["#1f77b4", "#d62728"]):
            ax[1].plot(range(1, M + 1), cur[v].mean(0), "o-", color=c, label=v)
        ax[1].set_xlabel("models in ensemble"); ax[1].set_ylabel("CV macro-F1"); ax[1].legend()
        ax[1].set_title("Madrid CV vs ensemble size"); ax[1].grid(alpha=.3)
        fig.tight_layout(); fig.savefig(OUT / "ensemble_report.png", dpi=150)
    except ImportError:
        pass
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--folds", default="")
    ap.add_argument("--final", action="store_true")
    ap.add_argument("--report", action="store_true")
    a = ap.parse_args()
    log = lambda s: print(s, flush=True)
    OUT.mkdir(parents=True, exist_ok=True)
    if a.folds or a.final:
        data = load()
        log(f"device={DEVICE} smoke={SMOKE} seeds={N_SEEDS}")
        for k in [int(x) for x in a.folds.split(",") if x.strip()]:
            run_fold(k, data, log)
        if a.final:
            run_final(data, log)
        log("WORKER DONE")
    if a.report:
        report()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/model_selection_final/transfer.py
"""
Transfer pipeline: build a fine-tuned building-age model for ANY dataset in the hackathon
parquet format, starting from the Madrid-trained TempCNN ensemble backbone.

    model = modelCreate(dataset, labels, centre_latlon=None, centre_xy=None, class_bounds=None,
                        backbone=".../backbone.pt")
    model.predictions        # every pixel: label, confidence tier, c1/c2, distances, margin, votes
    model.save(folder)

Nothing in here depends on WHICH city it is. Everything city-dependent is either an input
(file, labels, centre, class boundaries) or measured from the dataset itself:
  hemisphere  WRS-2 row of the dataset's own Landsat scenes (row 60 = equator for daytime
              scenes); cross-checked against the centre's latitude when one is given.
              Southern hemisphere -> acquisition DOYs shifted by half a year, so the backbone's
              "distance from summer" channel keeps its meaning (approach (a)).
  centre      input (lat/lon or projected x/y); fallback = centroid of the labelled pixels of
              the oldest class.
  channels    matched to the backbone BY NAME; missing ones = 0 after standardisation, and the
              first conv layer becomes an unfreezing candidate. Years matched BY CALENDAR YEAR;
              missing years = unobserved. Unknown sensors -> error.
  normalisation  the backbone's source statistics, then AdaBN on the dataset's unlabelled pixels.
  spatial scale  from the dataset's own (unlabelled) embeddings: similarity of pixel pairs vs
              distance; the neighbourhood ends where the curve stops falling (one-sided 95%).
              Gives the neighbour radius R, the vote weight per distance, the CV block size.

Fine-tuning (few-shot, gradual unfreezing; the "Freezable vs. Trainable" note):
  stage -1  backbone as is            stage 0  AdaBN only (BN statistics from the dataset)
  stage 1   + head                    stage 2  + dense layer (LR / eta)
  stage 3   + last conv block (LR / eta^2)     stage 4  + first conv block (LR / eta^3), only
                                                if some backbone channels are missing
  Frozen = requires_grad False AND not in the optimizer; every BatchNorm stays in eval mode
  (dataset statistics); dropout only in trained layers. Loss = SupCon, as in pretraining.
  Stage x LR {1e-4, 3e-4, 1e-3} x eta {3, 10} x epochs chosen by spatial-block CV on the labelled
  pixels, 1-SE rule toward the simplest model.
Metric learning: class prototypes (and kNN, reported) on the labelled pixels' embeddings.

Final label (prototype distances d = 1 - mean cosine over the ensemble; c1 closest, c2 second):
  margin d(c2) - d(c1) >= tau                         -> c1, HIGH
  else spatial neighbours within R (ground-truth labels + HIGH pixels, the latter weighted by
  their calibrated accuracy) vote between c1 and c2   -> winner, MODERATE-HIGH
  neighbours labelled but none is c1 / c2             -> c1, LOW, conflict flag
  no labelled neighbour                               -> c1, LOW
  tau: the margin quantile that maximises held-out macro-F1 of this rule in the same CV
  (fold models predict the held-out pixels AND their neighbours); fallback: Otsu split of
  the dataset's own margin distribution.

CLI:
  python transfer.py create --data X.parquet --labels L.csv [--centre LAT,LON] [--bounds 1960,1984,2004] --out DIR
  python transfer.py evaluate --data X.parquet --bounds B1,B2,B3 [--centre LAT,LON] --shots 5,10 --draws 10 --out DIR
"""
import argparse
import copy
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = Path(__file__).resolve().parent
for p in (HERE, HERE.parent / "model_selection_v3"):
    sys.path.insert(0, str(p))
import train_final as TF
from common_v3 import (BASE_BANDS, INDEX_COLS, META_CHANNELS, SENSORS, STATIC_COLS, label_weights,
                       preprocess_dataset, utm)

DEVICE = TF.DEVICE
NEIGH_CHANNELS = [f"neigh_{b}" for b in BASE_BANDS]
LRS, ETAS = (1e-4, 3e-4, 1e-3), (3, 10)
WRS_EQUATOR_ROW = 60
SMOKE = os.environ.get("MS_SMOKE") == "1"


def log(s):
    print(f"{time.strftime('%H:%M:%S')} {s}", flush=True)


# ══════════════════════════════════════════════════════════════════════════════
# Backbone
# ══════════════════════════════════════════════════════════════════════════════
def legacy_meta():
    """For backbone.pt files written before metadata was stored (ensemble run of 2026-11-10)."""
    return dict(channels=BASE_BANDS + INDEX_COLS + META_CHANNELS + NEIGH_CHANNELS, years=list(range(1984, 2026)),
                class_bounds=[1960, 1984, 2004], static_cols=list(STATIC_COLS), summer_doy_north=196,
                variant="best", n_spectral=11, n_meta=15, n_neigh=6)


def load_backbone(path):
    b = torch.load(path, map_location="cpu", weights_only=False)
    side = Path(path).with_name("backbone_meta.json")
    meta = b.get("meta") or (json.loads(side.read_text()) if side.exists() else legacy_meta())
    states = b["states"][meta["variant"]][:meta.get("n_members", len(b["states"][meta["variant"]]))]
    return dict(states=states, norm=b["norm"], arch=b["arch"], emb_dim=b["emb_dim"],
                n_channels=b["n_channels"], n_static=b["n_static"], meta=meta)


def reference_model(bb):
    a = bb["arch"]
    if "cfg" in a:                                       # sweep-style backbone
        import sweep
        return sweep.build_model(a["cfg"], bb["n_channels"], bb["n_static"])
    return TF.TempCNNEmb(bb["n_channels"], bb["n_static"], t_len=a["t_len"], filters=a["filters"],
                         kernel=a["kernel"], fc=a["fc"], emb=bb["emb_dim"])


class EnsembleNet(nn.Module):
    """All M ensemble members as ONE batched network (grouped convs, batched matmuls).
    Units: conv blocks 0..nb-1, dense = nb, head = nb+1. BatchNorm always uses running stats."""

    def __init__(self, states, ref):
        super().__init__()
        self.M, self.nb = len(states), len(ref.convs) // 4
        st = lambda k: torch.stack([s[k].float() for s in states]).to(DEVICE)
        self.attr, self.units = [], []
        for i in range(self.nb):
            conv, bn = ref.convs[4 * i], ref.convs[4 * i + 1]
            self.attr.append(dict(pad=conv.padding[0], dil=conv.dilation[0], mode=conv.padding_mode, eps=bn.eps))
            self.units.append(self._reg(f"u{i}", dict(w=st(f"convs.{4 * i}.weight"), b=st(f"convs.{4 * i}.bias"),
                                                      g=st(f"convs.{4 * i + 1}.weight"), be=st(f"convs.{4 * i + 1}.bias")),
                                        dict(rm=st(f"convs.{4 * i + 1}.running_mean"), rv=st(f"convs.{4 * i + 1}.running_var"))))
        self.bn_eps_fc = ref.fc[2].eps
        self.units.append(self._reg("dense", dict(w=st("fc.1.weight"), b=st("fc.1.bias"), g=st("fc.2.weight"),
                                                  be=st("fc.2.bias")),
                                    dict(rm=st("fc.2.running_mean"), rv=st("fc.2.running_var"))))
        self.units.append(self._reg("head", dict(w0=st("head.0.weight"), b0=st("head.0.bias"), w3=st("head.3.weight"),
                                                 b3=st("head.3.bias")), {}))
        self.p_drop = ref.drops[0].p
        self.trainable = set()
        self.templates = [{k: v.clone() for k, v in s.items()} for s in states]

    def _reg(self, name, params, buffers):
        d = nn.ParameterDict({k: nn.Parameter(v) for k, v in params.items()})
        setattr(self, f"p_{name}", d)
        for k, v in buffers.items():
            self.register_buffer(f"{name}_{k}", v)
        return (name, d)

    def set_trainable(self, units):
        self.trainable = set(units)
        for i, (_, d) in enumerate(self.units):
            for p in d.values():
                p.requires_grad_(i in self.trainable)

    def _drop(self, x, u):
        return F.dropout(x, self.p_drop, training=self.training and u in self.trainable)

    def run(self, h, static, start=0, stop=None):
        stop = self.nb + 2 if stop is None else stop
        M = self.M
        for u in range(start, stop):
            if u < self.nb:
                name, d = self.units[u]; a = self.attr[u]
                if u == 0 and start == 0:
                    B = h.shape[0]; x = h.transpose(1, 2).repeat(1, M, 1)               # shared input
                else:
                    B = h.shape[1]; x = h.permute(1, 0, 2, 3).reshape(B, -1, h.shape[-1])
                w = d["w"]; cout = w.shape[1]
                pad = a["pad"]
                if a["mode"] != "zeros" and pad > 0:
                    x = F.pad(x, (pad, pad), mode=a["mode"]); pad = 0
                y = F.conv1d(x, w.reshape(M * cout, w.shape[2], w.shape[3]), d["b"].reshape(-1), padding=pad,
                             dilation=a["dil"], groups=M)
                rm, rv = getattr(self, f"{name}_rm").reshape(1, -1, 1), getattr(self, f"{name}_rv").reshape(1, -1, 1)
                y = (y - rm) / torch.sqrt(rv + a["eps"]) * d["g"].reshape(1, -1, 1) + d["be"].reshape(1, -1, 1)
                y = self._drop(F.relu(y), u)
                h = y.reshape(B, M, cout, -1).permute(1, 0, 2, 3)
            elif u == self.nb:
                d = self.units[u][1]
                x = h.reshape(M, h.shape[1], -1)
                y = torch.baddbmm(d["b"].unsqueeze(1), x, d["w"].transpose(1, 2))
                y = (y - self.dense_rm.unsqueeze(1)) / torch.sqrt(self.dense_rv.unsqueeze(1) + self.bn_eps_fc) \
                    * d["g"].unsqueeze(1) + d["be"].unsqueeze(1)
                h = self._drop(F.relu(y), u)
            else:
                d = self.units[u][1]
                x = torch.cat([h, static.unsqueeze(0).expand(M, -1, -1)], dim=2)
                y = self._drop(F.relu(torch.baddbmm(d["b0"].unsqueeze(1), x, d["w0"].transpose(1, 2))), u)
                h = F.normalize(torch.baddbmm(d["b3"].unsqueeze(1), y, d["w3"].transpose(1, 2)), dim=2)
        return h

    def to_states(self):
        out = [{k: v.clone() for k, v in t.items()} for t in self.templates]
        keymap = {}
        for i in range(self.nb):
            keymap[f"u{i}"] = dict(w=f"convs.{4 * i}.weight", b=f"convs.{4 * i}.bias", g=f"convs.{4 * i + 1}.weight",
                                   be=f"convs.{4 * i + 1}.bias", rm=f"convs.{4 * i + 1}.running_mean",
                                   rv=f"convs.{4 * i + 1}.running_var")
        keymap["dense"] = dict(w="fc.1.weight", b="fc.1.bias", g="fc.2.weight", be="fc.2.bias",
                               rm="fc.2.running_mean", rv="fc.2.running_var")
        keymap["head"] = dict(w0="head.0.weight", b0="head.0.bias", w3="head.3.weight", b3="head.3.bias")
        for name, d in self.units:
            for k, sk in keymap[name].items():
                v = d[k] if k in d else getattr(self, f"{name}_{k}")
                for m in range(self.M):
                    out[m][sk] = v[m].detach().cpu().to(out[m][sk].dtype)
        return out


def adabn_states(bb, X, S):
    """Re-estimate every BatchNorm's statistics on the dataset's unlabelled pixels (Li et al. 2016)."""
    out = []
    perm = np.random.default_rng(0).permutation(len(X))
    for st in bb["states"]:
        m = reference_model(bb).to(DEVICE); m.load_state_dict(st)
        bns = [x for x in m.modules() if isinstance(x, nn.BatchNorm1d)]
        for x in bns:
            x.reset_running_stats(); x.momentum = None
        m.train()
        for x in m.modules():
            if isinstance(x, nn.Dropout):
                x.eval()
        with torch.no_grad():
            for i in range(0, len(perm), 4096):
                ii = perm[i:i + 4096]
                m(X[ii], S[ii])
        for x in bns:
            x.momentum = 0.1
        out.append({k: v.detach().cpu().clone() for k, v in m.state_dict().items()})
    return out


@torch.no_grad()
def embed_all(net, X, S, chunk=2048):
    net.eval()
    return torch.cat([net.run(X[i:i + chunk], S[i:i + chunk]).cpu() for i in range(0, len(X), chunk)], dim=1)


# ══════════════════════════════════════════════════════════════════════════════
# Dataset preparation (label-free)
# ══════════════════════════════════════════════════════════════════════════════
def hemisphere_from_scenes(path):
    import pyarrow.parquet as pq
    t = pq.read_table(path, columns=[f"scene_id_{s}" for s in (1, 2, 3)]).to_pandas()
    sid = pd.concat([t[c] for c in t.columns]).dropna()
    rows = sid.str.split("_").str[2].str[3:6].astype(int)
    row = float(np.median(rows))
    return ("north" if row < WRS_EQUATOR_ROW else "south"), row


def neigh_std(spec, px, py):
    """Std of each band across the 3x3 neighbourhood per year (identical to extra_features_v4)."""
    N, T = spec.shape[:2]
    gx, gy = px - px.min() + 1, py - py.min() + 1
    H, W = gy.max() + 2, gx.max() + 2
    out = np.empty((N, T, 6), np.float32)
    for j in range(6):
        g = np.full((T, H, W), np.nan, np.float32)
        g[:, gy, gx] = spec[:, :, j].T
        s1 = np.zeros((T, H, W), np.float32); s2 = np.zeros_like(s1); c = np.zeros_like(s1)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                sh = np.roll(np.roll(g, dy, axis=1), dx, axis=2)
                ok = ~np.isnan(sh)
                s1 += np.where(ok, sh, 0); s2 += np.where(ok, sh * sh, 0); c += ok
        with np.errstate(all="ignore"):
            var = s2 / c - (s1 / c) ** 2
        sd = np.where(c >= 3, np.sqrt(np.clip(var, 0, None)), np.nan)
        out[:, :, j] = sd[:, gy, gx].T
    return out


class City:
    """A dataset prepared for the backbone: standardised series, aligned channels and years."""

    def __init__(self, path, bb, centre_latlon=None, centre_xy=None, class_bounds=None):
        self.path, meta, norm = str(path), bb["meta"], bb["norm"]
        hemi, row = hemisphere_from_scenes(path)
        self.report = dict(hemisphere_from_wrs=hemi, median_wrs_row=row)
        if centre_latlon is not None:
            hemi_c = "north" if centre_latlon[0] >= 0 else "south"
            if hemi_c != hemi:
                self.report["hemisphere_warning"] = f"WRS rows say {hemi}, centre latitude says {hemi_c}; using the centre"
            hemi = hemi_c
        self.hemisphere = hemi
        doy_shift = 0.0 if hemi == "north" else 365.25 / 2
        feats, spec, metaa, stats = preprocess_dataset(path, centre_xy=None, class_bounds=class_bounds,
                                                       doy_shift=doy_shift, strict=False, keep_xy=True)
        self.stats = stats
        self.px, self.py = feats["px_key"].to_numpy(np.int64), feats["py_key"].to_numpy(np.int64)
        self.x, self.y = feats["pixel_x"].to_numpy(float), feats["pixel_y"].to_numpy(float)
        self.coverage = feats["coverage"].to_numpy(float)
        self.dataset_class = feats["age_class"].to_numpy()          # -1 unless class_bounds given (evaluation)
        neigh = neigh_std(spec, self.px, self.py)
        N = len(self.px)
        # standardise with the backbone's source statistics, then place into the backbone's layout
        z_spec = (spec - norm["spec_mu"]) / norm["spec_sd"]
        z_neigh = np.nan_to_num((neigh - norm["neigh_mu"]) / norm["neigh_sd"], nan=0.0)
        own = dict(zip(BASE_BANDS + INDEX_COLS + META_CHANNELS + NEIGH_CHANNELS,
                       list(np.moveaxis(z_spec, 2, 0)) + list(np.moveaxis(metaa, 2, 0)) + list(np.moveaxis(z_neigh, 2, 0))))
        years_own = stats["years"]
        yb = meta["years"]
        ymap = {y: i for i, y in enumerate(years_own)}
        seq = np.zeros((N, len(yb), len(meta["channels"])), np.float32)
        missing_ch = [c for c in meta["channels"] if c not in own]
        for ci, c in enumerate(meta["channels"]):
            if c in own:
                for ti, y in enumerate(yb):
                    if y in ymap:
                        seq[:, ti, ci] = own[c][:, ymap[y]]
        self.seq = torch.from_numpy(seq).to(DEVICE)
        self.report.update(missing_channels=missing_ch, extra_channels=[c for c in own if c not in meta["channels"]],
                           missing_years=[y for y in yb if y not in ymap], dropped_years=[y for y in years_own if y not in yb],
                           n_pixels=N, pixel_bands_filled_from_year_median=stats["pixel_bands_filled_from_year_median"])
        self.allow_first_conv = bool(missing_ch)
        self.centre_latlon, self.centre_xy = centre_latlon, centre_xy
        self.cache = {}
        # pixel grid (30 m units = px/py keys)
        self.gx, self.gy = self.px - self.px.min(), self.py - self.py.min()
        self.G = np.full((self.gy.max() + 1, self.gx.max() + 1), -1, np.int64)
        self.G[self.gy, self.gx] = np.arange(N)

    def resolve_centre(self, labels_idx=None, labels_cls=None):
        if self.centre_xy is not None:
            return tuple(self.centre_xy), "input (projected)"
        if self.centre_latlon is not None:
            lat, lon = self.centre_latlon
            z0 = int((lon + 180) // 6) + 1
            cands = []
            for z in (z0, z0 - 1, z0 + 1):
                cx, cy = utm(lat, lon, z)
                dx = max(self.x.min() - cx, 0, cx - self.x.max()); dy = max(self.y.min() - cy, 0, cy - self.y.max())
                cands.append((math.hypot(dx, dy), z, (cx, cy)))
            dist, z, xy = min(cands)
            if dist > 0:
                raise ValueError("the centre lies outside the dataset in every nearby UTM zone")
            return xy, f"input (lat/lon, UTM zone {z})"
        if labels_idx is None or len(labels_idx) == 0:
            raise ValueError("no centre given and no labels to fall back on")
        old = labels_idx[labels_cls == labels_cls.min()]
        return (float(self.x[old].mean()), float(self.y[old].mean())), \
            f"fallback: centroid of the {len(old)} labelled pixels of the oldest class"

    def static(self, centre_xy, bb):
        d = np.hypot(self.x - centre_xy[0], self.y - centre_xy[1])
        vals = dict(coverage=self.coverage, dist_centre_rel=d / np.percentile(d, 95))
        cols = bb["meta"]["static_cols"]
        tab = np.stack([vals[c] for c in cols], axis=1)
        return torch.from_numpy(((tab - bb["norm"]["static_mu"]) / bb["norm"]["static_sd"]).astype(np.float32)).to(DEVICE)


# ══════════════════════════════════════════════════════════════════════════════
# Spatial structure (label-free) and neighbour votes
# ══════════════════════════════════════════════════════════════════════════════
def ring_offsets(R):
    rings = []
    for d in range(1, R + 1):
        ys, xs = np.mgrid[-d:d + 1, -d:d + 1]
        m = np.rint(np.hypot(xs, ys)) == d
        rings.append(np.stack([ys[m], xs[m]], axis=1))
    return rings


def spatial_curve(city, Z, rng):
    """Mean similarity of pixel pairs at distance d (rings, 30 m units) from the dataset's own
    embeddings. Neighbourhood R = last ring before the curve stops falling (one-sided 95%);
    weight(d) = excess similarity over that plateau, scaled to 1 at d = 1."""
    N = len(city.px)
    anchors = rng.choice(N, min(N, 10_000), replace=False)
    Zt = Z / np.linalg.norm(Z, axis=1, keepdims=True)
    cap = max(2, min(city.G.shape) // 4)
    e, se = [], []
    ay, ax = city.gy[anchors], city.gx[anchors]
    for d, offs in enumerate(ring_offsets(cap), start=1):
        s_sum = np.zeros(len(anchors)); s_n = np.zeros(len(anchors))
        for dy, dx in offs:
            yy, xx = ay + dy, ax + dx
            ok = (yy >= 0) & (yy < city.G.shape[0]) & (xx >= 0) & (xx < city.G.shape[1])
            j = np.full(len(anchors), -1); j[ok] = city.G[yy[ok], xx[ok]]
            v = j >= 0
            s_sum[v] += np.einsum("ij,ij->i", Zt[anchors[v]], Zt[j[v]]); s_n[v] += 1
        have = s_n > 0
        vals = s_sum[have] / s_n[have]
        e.append(vals.mean()); se.append(vals.std(ddof=1) / math.sqrt(len(vals)))
        if d >= 2 and not (e[-2] - e[-1] > 1.645 * math.hypot(se[-2], se[-1])):
            break
    e, se = np.array(e), np.array(se)
    R = len(e) - 1                                        # last ring that was still falling
    plateau = e[-1]
    if R < 1 or e[0] <= plateau:
        return dict(R=0, weights=[], curve=e.tolist(), se=se.tolist(), plateau=float(plateau))
    w = np.clip((e[:R] - plateau) / (e[0] - plateau), 0, 1)
    return dict(R=int(R), weights=w.tolist(), curve=e.tolist(), se=se.tolist(), plateau=float(plateau))


class Neighbours:
    """Spatial neighbours within R of a set of pixels (ring index per offset)."""

    def __init__(self, city, pix, R):
        self.R = R
        offs = ring_offsets(R) if R > 0 else []
        self.ring = np.concatenate([np.full(len(o), d) for d, o in enumerate(offs)]) if offs else np.zeros(0, int)
        allo = np.concatenate(offs) if offs else np.zeros((0, 2), int)
        yy = city.gy[pix][:, None] + allo[None, :, 0]; xx = city.gx[pix][:, None] + allo[None, :, 1]
        ok = (yy >= 0) & (yy < city.G.shape[0]) & (xx >= 0) & (xx < city.G.shape[1])
        self.j = np.full(yy.shape, -1); self.j[ok] = city.G[yy[ok], xx[ok]]

    def votes(self, label, rel, K, w):
        """label/rel: per city pixel (-1 = unlabelled). Ring-normalised, distance-weighted votes."""
        n = self.j.shape[0]
        if self.R == 0:
            return np.zeros((n, K)), np.zeros(n, bool)
        jj = np.where(self.j >= 0, self.j, 0)
        lab = np.where(self.j >= 0, label[jj], -1)
        r = np.where(lab >= 0, rel[jj], 0.0)
        onehot = (lab[..., None] == np.arange(K)) * r[..., None]                    # (n, off, K)
        R1 = np.eye(self.R)[self.ring]                                               # (off, R)
        S = np.einsum("nok,or->nrk", onehot, R1); Nr = np.einsum("no,or->nr", r, R1)
        with np.errstate(invalid="ignore", divide="ignore"):
            frac = np.where(Nr[..., None] > 0, S / Nr[..., None], 0.0)
        return np.einsum("nrk,r->nk", frac, np.asarray(w)), (Nr > 0).any(axis=1)


def decide(c1, c2, high, votes, has_lab):
    """The confidence rule. Returns label, tier (3 high, 2 moderate-high, 1 low), conflict flag."""
    n = len(c1); idx = np.arange(n)
    v1, v2 = votes[idx, c1], votes[idx, c2]
    label = c1.copy(); tier = np.where(high, 3, 1); conflict = np.zeros(n, bool)
    low = ~high
    some = low & has_lab & (v1 + v2 > 0)
    label[some & (v2 > v1)] = c2[some & (v2 > v1)]
    tier[some] = 2
    conflict[low & has_lab & ~(v1 + v2 > 0)] = True
    return label, tier, conflict


# ══════════════════════════════════════════════════════════════════════════════
# Few-shot fine-tuning
# ══════════════════════════════════════════════════════════════════════════════
def balanced_batches(n_idx, y, per_class, K, rng):
    pools = [rng.permutation(n_idx[y[n_idx] == c]) for c in range(K)]
    ptr = [0] * K
    for _ in range(max(1, len(n_idx) // (per_class * K))):
        parts = []
        for c in range(K):
            if ptr[c] + per_class > len(pools[c]):
                pools[c], ptr[c] = rng.permutation(pools[c]), 0
            parts.append(pools[c][ptr[c]:ptr[c] + per_class]); ptr[c] += per_class
        yield rng.permutation(np.concatenate(parts))


def protos(z, y, w, K):
    """z (M, n, E) -> (M, K, E) label-weighted, normalised class means."""
    oh = F.one_hot(torch.as_tensor(y, device=z.device), K).float() * torch.as_tensor(np.asarray(w, np.float32), device=z.device)[:, None]
    P = torch.einsum("nk,mne->mke", oh, z) / oh.sum(0).clamp_min(1e-9)[None, :, None]
    return F.normalize(P, dim=2)


def ens_sims(z, P):
    return torch.einsum("mne,mke->nk", z, P) / z.shape[0]


def stage_units(stage, nb):
    return {1: [nb + 1], 2: [nb, nb + 1], 3: [nb - 1, nb, nb + 1], 4: [0, nb - 1, nb, nb + 1]}[stage]


def lr_mult(u, nb, eta):
    """head: LR; dense: LR/eta; last conv block: LR/eta^2; first conv block: LR/eta^3."""
    depth = {nb + 1: 0, nb: 1, nb - 1: 2, 0: 3}[u]
    return 1.0 if depth == 0 else 1.0 / eta ** depth


def finetune(states, ref, stage, lr, eta, H, S, y, w, K, epochs, seed, level, eval_cb=None):
    """Train the trainable top of the ensemble on cached activations H at `level`."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    net = EnsembleNet(states, ref)
    nb = net.nb
    units = stage_units(stage, nb)
    net.set_trainable(units)
    groups = [dict(params=list(net.units[u][1].values()), lr=lr * lr_mult(u, nb, eta)) for u in units]
    opt = torch.optim.AdamW(groups, weight_decay=TF.WEIGHT_DECAY)
    y_t, w_t = torch.as_tensor(y, device=DEVICE), torch.as_tensor(w, device=DEVICE).float()
    per_class = int(min(TF.FIXED["per_class"], np.bincount(y, minlength=K).min()))
    params = [p for g in groups for p in g["params"]]
    for ep in range(1, epochs + 1):
        net.train()
        for bi in balanced_batches(np.arange(len(y)), y, per_class, K, rng):
            bi_t = torch.as_tensor(bi, device=DEVICE)
            hb = H[bi_t] if level == 0 else H[:, bi_t]
            z = net.run(hb, S[bi_t], start=level)
            loss = sum(TF.supcon(z[m], y_t[bi_t], 1.0 / TF.TAU0, w_t[bi_t]) for m in range(net.M))
            opt.zero_grad(); loss.backward()
            sq = sum((p.grad.reshape(net.M, -1) ** 2).sum(1) for p in params)      # per-model clipping
            scale = (TF.FIXED["grad_clip"] / (sq.sqrt() + 1e-6)).clamp(max=1.0)
            for p in params:
                p.grad.mul_(scale.view(-1, *[1] * (p.grad.dim() - 1)))
            opt.step()
        if eval_cb is not None:
            net.eval()
            with torch.no_grad():
                eval_cb(ep, net)
    net.eval()
    return net


def macro_f1(y, p, K):
    from sklearn.metrics import f1_score
    return float(f1_score(y, p, average="macro", labels=range(K)))


def boot_se(y, p, K, rng, B=200):
    n = len(y)
    return float(np.std([macro_f1(y[s], p[s], K) for s in (rng.integers(0, n, n) for _ in range(B))]))


def otsu(v):
    h, edges = np.histogram(v, bins="auto")
    c = (edges[:-1] + edges[1:]) / 2
    best, thr = -1, float(np.median(v))
    for i in range(1, len(h)):
        w0, w1 = h[:i].sum(), h[i:].sum()
        if w0 == 0 or w1 == 0:
            continue
        m0, m1 = (h[:i] * c[:i]).sum() / w0, (h[i:] * c[i:]).sum() / w1
        b = w0 * w1 * (m0 - m1) ** 2
        if b > best:
            best, thr = b, edges[i]
    return thr


class TransferModel:
    def __init__(self, **kw):
        self.__dict__.update(kw)

    def save(self, folder):
        folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
        torch.save(dict(states=self.states, norm=self.norm, meta=self.backbone_meta, arch=self.arch,
                        classes=self.classes, config=self.config, calibration=self.calibration), folder / "transfer_model.pt")
        self.predictions.to_parquet(folder / "predictions.parquet", index=False)
        (folder / "transfer_report.json").write_text(json.dumps(self.report, indent=1, default=float))


# ══════════════════════════════════════════════════════════════════════════════
# modelCreate
# ══════════════════════════════════════════════════════════════════════════════
def _context(city, bb, centre_xy):
    """Label-free, per dataset and centre: AdaBN states, embeddings, spatial curve. Cached."""
    key = tuple(np.round(centre_xy, 3))
    if key in city.cache:
        return city.cache[key]
    S = city.static(centre_xy, bb)
    if "adabn" not in city.cache:
        t0 = time.time()
        city.cache["adabn"] = adabn_states(bb, city.seq, S)
        log(f"  AdaBN on {len(city.px):,} unlabelled pixels: {time.time() - t0:.0f}s")
    ref = reference_model(bb)
    base_net, ada_net = EnsembleNet(bb["states"], ref), EnsembleNet(city.cache["adabn"], ref)
    Zb, Za = embed_all(base_net, city.seq, S), embed_all(ada_net, city.seq, S)
    Zcat = (Za.permute(1, 0, 2).reshape(len(city.px), -1) / math.sqrt(Za.shape[0])).numpy()
    curve = spatial_curve(city, Zcat, np.random.default_rng(0))
    log(f"  spatial structure: neighbourhood R = {curve['R']} px ({30 * curve['R']} m)")
    ctx = dict(S=S, ref=ref, Zb=Zb, Za=Za, curve=curve, ada_net=ada_net)
    city.cache[key] = ctx
    return ctx


def modelCreate(data, labels, centre_latlon=None, centre_xy=None, class_bounds=None,
                backbone=HERE / "results" / "ensemble" / "backbone.pt", verbose=True):
    """data: parquet path or a prepared City. labels: DataFrame px_key, py_key and age_class
    (any integer classes) or year (+ class_bounds)."""
    t_start = time.time()
    bb = data.bb if isinstance(data, City) else load_backbone(backbone)
    city = data if isinstance(data, City) else City(data, bb, centre_latlon, centre_xy)
    city.bb = bb
    # ── labels ────────────────────────────────────────────────────────────────
    key = pd.Series(np.arange(len(city.px)), index=pd.MultiIndex.from_arrays([city.px, city.py]))
    li = key.reindex(pd.MultiIndex.from_arrays([labels["px_key"].to_numpy(), labels["py_key"].to_numpy()]))
    if li.isna().any():
        raise ValueError(f"{int(li.isna().sum())} labelled pixels are not in the dataset")
    sup = li.to_numpy(np.int64)
    if "age_class" in labels:
        raw_cls = labels["age_class"].to_numpy()
    elif "year" in labels and class_bounds is not None:
        raw_cls = np.searchsorted(np.asarray(class_bounds, float), labels["year"].to_numpy(float), side="left") + 1
    else:
        raise ValueError("labels need age_class, or year together with class_bounds")
    classes = np.unique(raw_cls)
    K = len(classes)
    y = np.searchsorted(classes, raw_cls)
    w = (label_weights(labels["year"].to_numpy(float), bounds=list(class_bounds))[0]
         if "year" in labels and class_bounds is not None else np.ones(len(sup), np.float32))
    centre, centre_src = city.resolve_centre(sup, raw_cls)
    ctx = _context(city, bb, centre)
    S, ref, curve = ctx["S"], ctx["ref"], ctx["curve"]
    M = len(bb["states"]); nb = ctx["ada_net"].nb
    E_max = 2 if SMOKE else TF.FIXED["max_epochs"]
    rng = np.random.default_rng(12345)
    # ── spatial-block folds on the labelled pixels ───────────────────────────
    from sklearn.model_selection import StratifiedGroupKFold
    n_min = int(np.bincount(y, minlength=K).min())
    n_splits = min(5, n_min)
    blk = max(curve["R"], 1)
    groups = (city.gx[sup] // blk) * 1_000_003 + city.gy[sup] // blk
    folds = list(StratifiedGroupKFold(n_splits, shuffle=True, random_state=0).split(np.zeros(len(y)), y, groups)) \
        if n_splits >= 2 else []
    stages = [1, 2, 3] + ([4] if city.allow_first_conv else [])
    configs = [(-1, None, None), (0, None, None)] + [(1, lr, None) for lr in LRS] + \
              [(s, lr, eta) for s in stages[1:] for lr in LRS for eta in ETAS]
    level_of = {1: nb, 2: nb, 3: nb - 1, 4: 0}
    level_of[1] = nb + 1
    sup_t = torch.as_tensor(sup, device=DEVICE)
    act_cache = {}

    def acts(level, idx_t):
        k = (level, int(idx_t.numel()), int(idx_t[0]) if idx_t.numel() else -1, int(idx_t.sum()))
        if k not in act_cache:
            with torch.no_grad():
                ctx["ada_net"].eval()
                act_cache[k] = city.seq[idx_t] if level == 0 else ctx["ada_net"].run(city.seq[idx_t], S[idx_t], 0, level)
        return act_cache[k]

    # ── CV over configurations ───────────────────────────────────────────────
    cv = {}                                    # (config, epoch) -> pooled held-out predictions
    if folds:
        for fi, (tr, te) in enumerate(folds):
            for cfg in configs:
                st, lr, eta = cfg
                if st in (-1, 0):
                    Z = ctx["Zb"] if st == -1 else ctx["Za"]
                    zs = Z[:, sup].to(DEVICE)
                    P = protos(zs[:, tr], y[tr], w[tr], K)
                    cv.setdefault((cfg, 0), np.full(len(y), -1))[te] = ens_sims(zs[:, te], P).argmax(1).cpu().numpy()
                    continue
                lev = level_of[st]
                H = acts(lev, sup_t)

                def cb(ep, net, tr=tr, te=te, cfg=cfg, H=H, lev=lev):
                    hb = (lambda ii: H[torch.as_tensor(ii, device=DEVICE)] if lev == 0 else H[:, torch.as_tensor(ii, device=DEVICE)])
                    ztr = net.run(hb(tr), S[sup_t[tr]], start=lev); zte = net.run(hb(te), S[sup_t[te]], start=lev)
                    P = protos(ztr, y[tr], w[tr], K)
                    cv.setdefault((cfg, ep), np.full(len(y), -1))[te] = ens_sims(zte, P).argmax(1).cpu().numpy()

                Htr = H[torch.as_tensor(tr, device=DEVICE)] if lev == 0 else H[:, torch.as_tensor(tr, device=DEVICE)]
                finetune(city.cache["adabn"], ref, st, lr, eta, Htr, S[sup_t[tr]],
                         y[tr], w[tr], K, E_max, seed=fi, level=lev, eval_cb=cb)
        scores = {k: macro_f1(y, p, K) for k, p in cv.items() if (p >= 0).all()}
        order = lambda k: ({-1: 0, 0: 1}.get(k[0][0], k[0][0] + 1), k[1], k[0][1] or 0, -(k[0][2] or 0))
        best_k = max(scores, key=lambda k: (scores[k], -order(k)[0]))
        se = boot_se(y, cv[best_k], K, rng)
        chosen = min((k for k in scores if scores[k] >= scores[best_k] - se), key=order)
        sel = dict(best=dict(config=best_k[0], epochs=best_k[1], f1=scores[best_k]), se=se,
                   chosen=dict(config=chosen[0], epochs=chosen[1], f1=scores[chosen]), n_folds=len(folds),
                   n_candidates=len(scores))
    else:
        chosen = ((0, None, None), 0)
        sel = dict(chosen=dict(config=chosen[0], epochs=0, f1=None), note="fewer than 2 labels in some class: no CV")
    (st, lr, eta), E = chosen
    if verbose:
        log(f"  selected: stage {st}, lr {lr}, eta {eta}, epochs {E} (CV macro-F1 {sel['chosen']['f1']})")

    # ── fit on a set of labelled pixels and embed the whole dataset ──────────
    def fit_and_embed(tr):
        if st in (-1, 0):
            Z = ctx["Zb"] if st == -1 else ctx["Za"]
            return Z, (bb["states"] if st == -1 else city.cache["adabn"])
        lev = level_of[st]
        H = acts(lev, sup_t)
        Htr = H[torch.as_tensor(tr, device=DEVICE)] if lev == 0 else H[:, torch.as_tensor(tr, device=DEVICE)]
        net = finetune(city.cache["adabn"], ref, st, lr, eta, Htr, S[sup_t[tr]], y[tr], w[tr], K, E, seed=999, level=lev)
        out = []
        with torch.no_grad():
            for i in range(0, len(city.px), 2048):
                ii = torch.arange(i, min(i + 2048, len(city.px)), device=DEVICE)
                h = city.seq[ii] if lev == 0 else ctx["ada_net"].run(city.seq[ii], S[ii], 0, lev)
                out.append(net.run(h, S[ii], start=lev).cpu())
        return torch.cat(out, dim=1), net.to_states()

    def margins(Z, tr):
        P = protos(Z[:, sup[tr]].to(DEVICE), y[tr], w[tr], K)
        sims = torch.cat([ens_sims(Z[:, i:i + 8192].to(DEVICE), P).cpu() for i in range(0, Z.shape[1], 8192)])
        d = 1 - sims.numpy()
        o = np.argsort(d, axis=1)
        c1, c2 = o[:, 0], o[:, 1]
        idx = np.arange(len(d))
        delta = d[idx, c2] - d[idx, c1]
        q = (np.argsort(np.argsort(delta)) + 0.5) / len(delta)                 # margin quantile
        return d, c1, c2, delta, q

    # ── calibration: reliability of c1 by margin, and tau, from the same folds ──
    from sklearn.isotonic import IsotonicRegression
    calib = dict(source=None)
    R, wts = curve["R"], curve["weights"]
    if folds:
        held = []                                                               # per fold: data for tau search
        for fi, (tr, te) in enumerate(folds):
            Zf, _ = fit_and_embed(tr)
            d, c1, c2, delta, q = margins(Zf, tr)
            held.append((tr, te, c1, c2, q))
        q_te = np.concatenate([h[4][sup[h[1]]] for h in held])
        ok_te = np.concatenate([h[2][sup[h[1]]] == y[h[1]] for h in held])
        iso = IsotonicRegression(increasing=True, y_min=0, y_max=1, out_of_bounds="clip").fit(q_te, ok_te) \
            if len(np.unique(q_te)) > 1 else None
        acc0 = float(ok_te.mean())
        rel_fn = lambda qq: (np.zeros(0) if len(qq) == 0 else iso.predict(qq) if iso is not None
                             else np.full(len(qq), acc0))
        grid = np.round(np.arange(0, 1.0001, 0.01), 2)
        f1_q, preds_q = [], []
        if R > 0:
            nbrs = [Neighbours(city, sup[te], R) for (tr, te, *_r) in held]
            for qq in grid:
                yt, yp = [], []
                for (tr, te, c1, c2, q), nb_ in zip(held, nbrs):
                    lab = np.full(len(city.px), -1); rel = np.zeros(len(city.px))
                    hi = q >= qq
                    lab[hi] = c1[hi]; rel[hi] = rel_fn(q[hi])
                    lab[sup[te]] = -1; rel[sup[te]] = 0                     # held-out labels hidden
                    lab[sup[tr]] = y[tr]; rel[sup[tr]] = 1.0                # ground truth of the fold
                    v, hl = nb_.votes(lab, rel, K, wts)
                    lbl, _, _ = decide(c1[sup[te]], c2[sup[te]], q[sup[te]] >= qq, v, hl)
                    yt.append(y[te]); yp.append(lbl)
                f1_q.append(macro_f1(np.concatenate(yt), np.concatenate(yp), K)); preds_q.append(np.concatenate(yp))
            f1_q = np.array(f1_q)
            i_best = int(np.argmax(f1_q))
            q_star = float(grid[i_best])                                        # ties -> smallest quantile
            # cautious variant (reported alongside): smallest quantile within one bootstrap SE of the best,
            # i.e. hand pixels to the neighbours only as far as the CV gain exceeds its noise
            se_q = boot_se(np.concatenate(yt), preds_q[i_best], K, rng)
            q_1se = float(grid[int(np.nonzero(f1_q >= f1_q[i_best] - se_q)[0][0])])
            calib.update(source="CV (held-out labelled pixels)", q_star=q_star, q_cautious=q_1se, se=se_q,
                         f1_by_quantile=f1_q.tolist(), f1_model_only=float(f1_q[0]), f1_at_q_star=float(f1_q.max()))
        calib["iso_x"] = iso.X_thresholds_.tolist() if iso is not None else None
        calib["iso_y"] = iso.y_thresholds_.tolist() if iso is not None else None
    else:
        rel_fn = None
    # ── final model on all labelled pixels ───────────────────────────────────
    all_tr = np.arange(len(y))
    Zf, states_final = fit_and_embed(all_tr)
    d, c1, c2, delta, q = margins(Zf, all_tr)
    if calib.get("q_star") is not None:
        tau = float(np.quantile(delta, calib["q_star"]))
    else:
        tau = float(otsu(delta)); calib["source"] = "Otsu split of the dataset's margins (no usable CV)"
    if rel_fn is None:
        rel_fn = lambda qq: np.ones(len(qq))
    calib["tau"] = tau

    def apply_rule(tau_):
        high = delta >= tau_
        lab = np.full(len(city.px), -1); rel = np.zeros(len(city.px))
        lab[high] = c1[high]; rel[high] = rel_fn(q[high])
        lab[sup] = y; rel[sup] = 1.0
        low_idx = np.nonzero(~high)[0]
        votes = np.zeros((len(city.px), K)); has_lab = np.zeros(len(city.px), bool)
        if R > 0:
            for i in range(0, len(low_idx), 2000):
                ii = low_idx[i:i + 2000]
                v, hl = Neighbours(city, ii, R).votes(lab, rel, K, wts)
                votes[ii], has_lab[ii] = v, hl
        label, tier, conflict = decide(c1, c2, high, votes, has_lab)
        label[sup] = y; tier[sup] = 4
        return label, tier, conflict, votes, has_lab

    label, tier, conflict, votes, has_lab = apply_rule(tau)
    if calib.get("q_cautious") is not None and calib["q_cautious"] != calib.get("q_star"):
        calib["tau_cautious"] = float(np.quantile(delta, calib["q_cautious"]))
        label_c = apply_rule(calib["tau_cautious"])[0]
    else:
        calib["tau_cautious"] = tau; label_c = label
    # kNN on the labelled pixels (reported)
    kk = int(math.sqrt(n_min)) | 1
    Zs = Zf[:, sup].to(DEVICE)
    knn = []
    for i in range(0, Zf.shape[1], 4096):
        sims = torch.einsum("mne,mse->ns", Zf[:, i:i + 4096].to(DEVICE), Zs) / Zf.shape[0]
        top = sims.topk(min(kk, len(sup)), dim=1).indices
        knn.append(F.one_hot(torch.as_tensor(y, device=DEVICE)[top], K).sum(1).argmax(1).cpu())
    knn = torch.cat(knn).numpy()
    tiers = np.array(["", "low", "moderate-high", "high", "labelled"])
    pred = pd.DataFrame(dict(px_key=city.px, py_key=city.py, label=classes[label], confidence=tiers[tier],
                             conflict=conflict, c1=classes[c1], c2=classes[c2], dist_c1=d[np.arange(len(d)), c1],
                             dist_c2=d[np.arange(len(d)), c2], margin=delta, margin_quantile=q,
                             vote_c1=votes[np.arange(len(d)), c1], vote_c2=votes[np.arange(len(d)), c2],
                             labelled_neighbour=has_lab, knn_label=classes[knn], prototype_label=classes[c1],
                             label_cautious=classes[label_c]))
    report = dict(city=city.report, centre=dict(xy=centre, source=centre_src), n_labels=int(len(sup)),
                  classes=classes.tolist(), label_counts=np.bincount(y, minlength=K).tolist(),
                  spatial=dict(R_px=R, R_m=30 * R, weights=wts, curve=curve["curve"]), selection=sel,
                  calibration={k: v for k, v in calib.items() if k not in ("iso_x", "iso_y")},
                  tiers={str(t): int((tier == i).sum()) for i, t in enumerate(tiers) if t},
                  conflicts=int(conflict.sum()), knn_k=kk, seconds=time.time() - t_start)
    if verbose:
        log(f"  tau source: {calib['source']}; tiers {report['tiers']}; {report['seconds']:.0f}s")
    return TransferModel(states=states_final, norm=bb["norm"], backbone_meta=bb["meta"], arch=bb["arch"],
                         classes=classes.tolist(), config=dict(stage=st, lr=lr, eta=eta, epochs=E), calibration=calib,
                         predictions=pred, report=report, support=sup, Z=Zf)


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation on a labelled dataset (simulated few-shot)
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(path, bounds, centre_latlon, shots, draws, out, backbone):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    bb = load_backbone(backbone)
    city = City(path, bb, centre_latlon=centre_latlon, class_bounds=bounds)
    city.bb = bb
    yr = pd.read_parquet(path, columns=["px_key", "py_key", "weighted_mean_year"]).groupby(["px_key", "py_key"]).first()
    years = yr.reindex(pd.MultiIndex.from_arrays([city.px, city.py]))["weighted_mean_year"].to_numpy(float)
    y_all = city.dataset_class
    K = int(y_all.max())
    same_scheme = list(bounds) == list(bb["meta"]["class_bounds"])
    log(f"evaluate {path}: {len(y_all):,} pixels, classes {np.bincount(y_all)[1:].tolist()}, "
        f"hemisphere {city.hemisphere}; class scheme {'matches' if same_scheme else 'differs from'} the backbone's")
    for k in shots:
        for r in range(draws):
            f = out / f"k{k}_d{r}.json"
            if f.exists():
                continue
            rng = np.random.default_rng(10_000 * k + r)                       # same draws as v3/v4
            sup = np.concatenate([rng.choice(np.nonzero(y_all == c)[0], k, replace=False) for c in range(1, K + 1)])
            labels = pd.DataFrame(dict(px_key=city.px[sup], py_key=city.py[sup], year=years[sup]))
            log(f"k={k} draw {r}")
            m = modelCreate(city, labels, class_bounds=bounds)
            pr = m.predictions
            yq = np.ones(len(y_all), bool); yq[sup] = False
            R = m.report["spatial"]["R_px"]
            from scipy.spatial import cKDTree
            dist, _ = cKDTree(np.c_[city.gx[sup], city.gy[sup]]).query(np.c_[city.gx, city.gy])
            far = yq & (dist > R)
            res = dict(k=k, draw=r, R_px=R, config=m.config, selection=m.report["selection"],
                       calibration=m.report["calibration"], tiers=m.report["tiers"], seconds=m.report["seconds"])
            ytrue = y_all
            for name, mask in (("all", yq), ("beyond_R", far)):
                blk = dict(n=int(mask.sum()))
                for col in ("label", "label_cautious", "prototype_label", "knn_label"):
                    blk[col] = macro_f1(ytrue[mask] - 1, pr[col].to_numpy()[mask] - 1, K)
                blk["accuracy"] = float((pr["label"].to_numpy()[mask] == ytrue[mask]).mean())
                for t in ("high", "moderate-high", "low"):
                    tm = mask & (pr["confidence"].to_numpy() == t)
                    blk[f"share_{t}"] = float(tm.sum() / max(mask.sum(), 1))
                    blk[f"acc_{t}"] = float((pr["label"].to_numpy()[tm] == ytrue[tm]).mean()) if tm.any() else None
                res[name] = blk
            # baseline: backbone as is, prototypes from the labelled pixels (the v4 few-shot protocol)
            ctx = city.cache[tuple(np.round(m.report["centre"]["xy"], 3))]
            Zb = ctx["Zb"]
            yi = np.searchsorted(np.array(m.classes), y_all[sup])
            P = protos(Zb[:, sup].to(DEVICE), yi, np.ones(len(sup)), K)
            pb = torch.cat([ens_sims(Zb[:, i:i + 8192].to(DEVICE), P).argmax(1).cpu() for i in range(0, Zb.shape[1], 8192)]).numpy()
            res["baseline_backbone_prototypes"] = {n: macro_f1(ytrue[mk] - 1, pb[mk], K) for n, mk in (("all", yq), ("beyond_R", far))}
            f.write_text(json.dumps(res, indent=1, default=float))
            log(f"  -> macro-F1 all {res['all']['label']:.4f} (prototype {res['all']['prototype_label']:.4f}, "
                f"backbone baseline {res['baseline_backbone_prototypes']['all']:.4f}) | beyond R {res['beyond_R']['label']:.4f}")
    summarise(out)


def summarise(out):
    out = Path(out)
    J = [json.loads(f.read_text()) for f in sorted(out.glob("k*_d*.json"))]
    if not J:
        return
    df = pd.DataFrame([dict(k=j["k"], draw=j["draw"], final=j["all"]["label"], cautious=j["all"]["label_cautious"],
                            proto=j["all"]["prototype_label"],
                            knn=j["all"]["knn_label"], base=j["baseline_backbone_prototypes"]["all"],
                            final_far=j["beyond_R"]["label"], base_far=j["baseline_backbone_prototypes"]["beyond_R"],
                            share_high=j["all"]["share_high"], share_mod=j["all"]["share_moderate-high"],
                            share_low=j["all"]["share_low"], acc_high=j["all"]["acc_high"],
                            acc_mod=j["all"]["acc_moderate-high"], acc_low=j["all"]["acc_low"],
                            stage=j["config"]["stage"], R=j["R_px"], tau_src=j["calibration"]["source"]) for j in J])
    g = df.groupby("k")
    L = ["# Transfer evaluation (simulated few-shot)\n",
         "Macro-F1 on all non-labelled pixels, mean ± sd over draws. *Backbone baseline* = Madrid backbone as is, "
         "prototypes from the labelled pixels. *Prototype* = fine-tuned model, closest prototype. *Final* = with the "
         "confidence/neighbour rule. *Beyond R* = only pixels farther than the neighbourhood radius from every label.\n",
         "| k / class | draws | Backbone baseline | Fine-tuned prototype | kNN | **Final** | Final − baseline | Cautious rule | Final, beyond R | Baseline, beyond R | Stages chosen |",
         "|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|"]
    for k, x in g:
        L.append(f"| {k} | {len(x)} | {x.base.mean():.4f} ± {x.base.std():.4f} | {x.proto.mean():.4f} | {x.knn.mean():.4f} | "
                 f"**{x.final.mean():.4f}** ± {x.final.std():.4f} | {(x.final - x.base).mean():+.4f} | {x.cautious.mean():.4f} | {x.final_far.mean():.4f} | "
                 f"{x.base_far.mean():.4f} | {x.stage.value_counts().to_dict()} |")
    L += ["\n## Confidence tiers (share of pixels / accuracy)\n",
          "| k / class | high | moderate-high | low | acc high | acc moderate-high | acc low | R (px) |", "|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for k, x in g:
        L.append(f"| {k} | {x.share_high.mean():.2f} | {x.share_mod.mean():.2f} | {x.share_low.mean():.2f} | "
                 f"{x.acc_high.mean():.3f} | {x.acc_mod.mean():.3f} | {x.acc_low.mean():.3f} | {x.R.median():.0f} |")
    (out / "transfer_eval_report.md").write_text("\n".join(L) + "\n")
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["create", "evaluate", "summary"])
    ap.add_argument("--data"); ap.add_argument("--labels"); ap.add_argument("--out", required=True)
    ap.add_argument("--centre", help="LAT,LON"); ap.add_argument("--centre-xy", help="X,Y in the dataset's projection")
    ap.add_argument("--bounds", help="class boundary years, e.g. 1960,1984,2004")
    ap.add_argument("--shots", default="5,10,25,50,100"); ap.add_argument("--draws", type=int, default=10)
    ap.add_argument("--backbone", default=str(HERE / "results" / "ensemble" / "backbone.pt"))
    a = ap.parse_args()
    parse = lambda s, t=float: None if s is None else tuple(t(v) for v in s.split(","))
    if a.cmd == "summary":
        summarise(a.out); return
    if a.cmd == "create":
        m = modelCreate(a.data, pd.read_csv(a.labels), centre_latlon=parse(a.centre), centre_xy=parse(a.centre_xy),
                        class_bounds=parse(a.bounds), backbone=a.backbone)
        m.save(a.out); log(f"saved to {a.out}")
    else:
        evaluate(a.data, parse(a.bounds, int), parse(a.centre), parse(a.shots, int), a.draws, a.out, a.backbone)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/model_selection_final/rebuild_prep.py
"""
Ground-up rebuild, step 1: everything the backbone needs, from the raw Madrid parquet only.
  cache/madrid_features_v3.parquet, madrid_seq_spec_v3.npy, madrid_seq_meta_v3.npz   (v3 rules)
  results/splits_spatial10.npz                                                       (10 spatial folds)
  extra/madrid_seq_extra_v4.npz  (key `neigh`: 3x3 neighbourhood std per band and year)
Prints array checksums next to the ones of the previous build (identical = same inputs).
Env: MS_CACHE_DIR, MS_EXTRA_DIR, MS_RESULTS_DIR. Raw data: <repo>/hackathon/data/madrid_train.parquet
"""
import hashlib
import os
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent / "model_selection_v3"))
from common_v3 import CACHE_DIR, RESULTS_DIR, preprocess_city_v3

EXTRA_DIR = Path(os.environ.get("MS_EXTRA_DIR", HERE.parent / "model_selection_v4" / "cache"))
N_FOLDS, BLOCK_PX, SEED = 10, 33, 42                      # as model_selection_v3/00_preprocess_v3.py
PREVIOUS = {'spec': 'a6d5e5a8f393cafb5283f27be690f703', 'meta': '62addf64c503f3c1874e6dbc2839e9fa',
            'age_class': 'f0c2dd09b2333dccf3c050a752e1559c', 'label_weight': '1813caa8c4b4de0f872d197419072d16',
            'static': 'ba97eaa5ae559c14ccdcd833d0465fcb', 'fold_id': '29ad5552c394351e6253764b02d7e6c5',
            'es_bits': '0b87d78df47e21530317f711394cfb1b', 'neigh': 'ee312d5ee15dc260e94edea08002863a'}


def splits(px, py, y0):
    groups = ((px - px.min()) // BLOCK_PX) * 100_000 + (py - py.min()) // BLOCK_PX
    fold_id = np.empty(len(y0), np.int8)
    es_bits = np.zeros(len(y0), np.uint16)
    outer = StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED)
    for k, (tr, va) in enumerate(outer.split(np.zeros(len(y0)), y0, groups)):
        fold_id[va] = k
        inner = StratifiedGroupKFold(7, shuffle=True, random_state=SEED + k)
        _, es_rel = next(inner.split(np.zeros(len(tr)), y0[tr], groups[tr]))
        es_bits[tr[es_rel]] |= np.uint16(1 << k)
    return fold_id, es_bits


def neigh_std(spec, px, py):
    N, T = spec.shape[:2]
    gx, gy = px - px.min() + 1, py - py.min() + 1
    H, W = gy.max() + 2, gx.max() + 2
    out = np.empty((N, T, 6), np.float32)
    for j in range(6):
        g = np.full((T, H, W), np.nan, np.float32)
        g[:, gy, gx] = spec[:, :, j].T
        s1 = np.zeros((T, H, W), np.float32); s2 = np.zeros_like(s1); c = np.zeros_like(s1)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                sh = np.roll(np.roll(g, dy, axis=1), dx, axis=2)
                ok = ~np.isnan(sh)
                s1 += np.where(ok, sh, 0); s2 += np.where(ok, sh * sh, 0); c += ok
        with np.errstate(all="ignore"):
            var = s2 / c - (s1 / c) ** 2
        sd = np.where(c >= 3, np.sqrt(np.clip(var, 0, None)), np.nan)
        out[:, :, j] = sd[:, gy, gx].T
    return out


def main():
    t0 = time.time()
    for d in (CACHE_DIR, RESULTS_DIR, EXTRA_DIR):
        d.mkdir(parents=True, exist_ok=True)
    feats, spec, meta, st = preprocess_city_v3("Madrid")
    print(f"Madrid preprocessed: {len(feats):,} pixels, {spec.shape[1]} years ({time.time() - t0:.0f}s)", flush=True)
    px, py = feats["px_key"].to_numpy(np.int64), feats["py_key"].to_numpy(np.int64)
    fold_id, es_bits = splits(px, py, feats["age_class"].to_numpy() - 1)
    neigh = neigh_std(spec, px, py).astype(np.float16)
    feats.to_parquet(CACHE_DIR / "madrid_features_v3.parquet", index=False)
    np.save(CACHE_DIR / "madrid_seq_spec_v3.npy", spec)
    np.savez_compressed(CACHE_DIR / "madrid_seq_meta_v3.npz", meta=meta.astype(np.float16))
    np.savez(RESULTS_DIR / "splits_spatial10.npz", fold_id=fold_id, es_bits=es_bits)
    np.savez_compressed(EXTRA_DIR / "madrid_seq_extra_v4.npz", neigh=neigh)
    h = lambda a: hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()
    now = dict(spec=h(spec), meta=h(meta.astype(np.float16)), age_class=h(feats["age_class"].to_numpy()),
               label_weight=h(feats["label_weight"].to_numpy()),
               static=h(feats[["coverage", "dist_centre_rel"]].to_numpy()), fold_id=h(fold_id), es_bits=h(es_bits),
               neigh=h(neigh))
    for k, v in now.items():
        print(f"  {k:13s} {'identical to previous build' if v == PREVIOUS[k] else 'DIFFERENT from previous build'}")
    print(f"step 1 done in {time.time() - t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/model_selection_final/choose_members.py
"""
Ground-up rebuild, step 3: apply the pre-registered rules to THIS run's Madrid CV (ensemble_final.py
fold JSONs) and write backbone_meta.json next to backbone.pt.
  variant   'swa' only if the full-ensemble SWA beats 'best' by > ROPE with P >= 0.95, else 'best'
  members   smallest M equivalent to the full ensemble (P(|diff| < ROPE) >= 0.95) AND better than the
            average single model by > ROPE with P >= 0.95; otherwise all members
Usage: python choose_members.py <results/ensemble dir>
"""
import json
import math
import sys
from pathlib import Path

import numpy as np
from scipy import stats

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
ROPE = 0.005


def corr_t(d, k=10):
    rho = 1 / k
    sd = math.sqrt((1 / len(d) + rho / (1 - rho)) * np.var(d, ddof=1))
    return stats.t(df=len(d) - 1, loc=float(np.mean(d)), scale=max(sd, 1e-9))


def main(out):
    out = Path(out)
    J = [json.loads((out / f"fold{k}.json").read_text()) for k in range(10)]
    cur = {v: np.array([j["curve"][v] for j in J]) for v in ("best", "swa")}
    t = corr_t(cur["swa"][:, -1] - cur["best"][:, -1])
    variant = "swa" if 1 - t.cdf(ROPE) >= 0.95 else "best"
    c = cur[variant]; N = c.shape[1]
    single = np.array([np.mean(j["single_f1"][variant]) for j in J])
    rows, chosen = [], N
    for M in range(1, N + 1):
        te, ts = corr_t(c[:, M - 1] - c[:, -1]), corr_t(c[:, M - 1] - single)
        p_eq, p_gain = float(te.cdf(ROPE) - te.cdf(-ROPE)), float(1 - ts.cdf(ROPE))
        rows.append(dict(M=M, f1=float(c[:, M - 1].mean()), p_equiv_full=p_eq, p_better_than_single=p_gain))
        if chosen == N and p_eq >= 0.95 and p_gain >= 0.95:
            chosen = M
    import transfer
    meta = transfer.legacy_meta()
    meta.update(variant=variant, n_members=chosen,
                n_members_rule="smallest M equivalent to the full ensemble (ROPE 0.005, P>=0.95) and better than "
                               "the average single model by > ROPE (P>=0.95); Madrid 10-fold CV of this build",
                ensemble_curve=rows, swa_minus_best=float(t.mean()))
    (out / "backbone_meta.json").write_text(json.dumps(meta, indent=1))
    print(f"variant {variant} (SWA - best {t.mean():+.4f}); members {chosen} of {N}")
    for r in rows:
        print(f"  M={r['M']:2d} F1 {r['f1']:.4f}  P(equiv full) {r['p_equiv_full']:.2f}  P(> single) {r['p_better_than_single']:.2f}")


if __name__ == "__main__":
    main(sys.argv[1])


In [ ]:
r = subprocess.run(PY + [f'{WD}/model_selection_final/rebuild_prep.py'], env=ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:]); assert r.returncode == 0

In [ ]:
E = dict(ENV, MS_RUN_NAME='ensemble', MS_N_SEEDS='10')
ens = f'{WD}/model_selection_final/ensemble_final.py'
run_parallel([([ens, '--folds', '0,2,4,6,8'], E), ([ens, '--folds', '1,3,5,7,9'], E), ([ens, '--final'], E)], 'ensemble')
r = subprocess.run(PY + [ens, '--report'], env=E, capture_output=True, text=True); print(r.stdout[-6000:], r.stderr[-2000:])

In [ ]:
r = subprocess.run(PY + [f'{WD}/model_selection_final/choose_members.py', f'{WD}/results/ensemble'], env=ENV,
                   capture_output=True, text=True, cwd=f'{WD}/model_selection_final'); print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0

In [ ]:
OUT = f'{WD}/results/transfer_eval_amsterdam'
tr = f'{WD}/model_selection_final/transfer.py'
A = ['--data', f'{WD}/hackathon/data/amsterdam_data.parquet', '--bounds', '1945,1984,2004', '--centre', '52.3731,4.8932',
     '--draws', '10', '--out', OUT, '--backbone', f'{WD}/results/ensemble/backbone.pt']
run_parallel([([tr, 'evaluate', '--shots', '100,10,5'] + A, ENV), ([tr, 'evaluate', '--shots', '50,25'] + A, ENV)], 'transfer')
r = subprocess.run(PY + [tr, 'summary', '--out', OUT], env=ENV, capture_output=True, text=True); print(r.stdout, r.stderr[-2000:])

In [ ]:
pack = f'{WD}/pack'; os.makedirs(pack, exist_ok=True)
shutil.copytree(f'{WD}/logs', f'{pack}/logs', dirs_exist_ok=True)
shutil.copytree(f'{WD}/results/transfer_eval_amsterdam', f'{pack}/transfer_eval_amsterdam', dirs_exist_ok=True)
os.makedirs(f'{pack}/ensemble', exist_ok=True)
for f in glob.glob(f'{WD}/results/ensemble/*'):
    if not f.endswith('_oof.npz'): shutil.copy(f, f'{pack}/ensemble/')
shutil.make_archive(f'{WD}/rebuild_results', 'zip', pack)
print(sorted(os.listdir(f'{pack}/ensemble')))